# Task
Explain the mathematical formulation and implement in PyTorch, with detailed comments, the following components: token embedding, sinusoidal positional encoding, multi-head self-attention (including masked attention), layer normalization, and a feed-forward network. Then, integrate residual connections and assemble these components into a decoder block. Finally, combine all implemented components, along with a final linear output layer, to build a complete decoder-only Transformer model in PyTorch. Provide a summary of the complete implementation and its key components.

## Token Embedding

### Subtask:
Explain the mathematical formulation of token embeddings and implement a PyTorch layer for creating them, including detailed comments.


```markdown
## Mathematical Formulation of Token Embeddings

Token embeddings are a fundamental concept in natural language processing (NLP) that allow us to represent discrete textual units (like words, subwords, or characters) as dense, continuous vector representations in a lower-dimensional space. This transformation is crucial because machine learning models typically operate on numerical data, and discrete tokens themselves do not inherently carry numerical meaning that can be easily processed.

Mathematically, a token embedding can be understood as a function, let's call it $E$, that maps each unique token $t$ from a vocabulary $V$ to a real-valued vector $v \in \mathbb{R}^d$, where $d$ is the embedding dimension.

$$ E: V \rightarrow \mathbb{R}^d $$

Here's a breakdown of the key components:

1.  **Vocabulary ($V$):** This is the set of all unique tokens present in our dataset. Each token is assigned a unique integer ID (index).

2.  **Embedding Dimension ($d$):** This is a hyperparameter that determines the size of the vector representation for each token. A higher dimension allows for more nuanced representation but also increases computational cost and model complexity. Typical dimensions range from 50 to 500, or even more.

3.  **Embedding Matrix (or Lookup Table):** The core of token embedding is an embedding matrix, often denoted as $\mathbf{W}_{emb} \in \mathbb{R}^{|V| \times d}$. Each row in this matrix corresponds to a unique token in the vocabulary, and that row is the $d$-dimensional vector representation (embedding) for that token.
    *   If a token $t_i$ has an integer ID of $i$, then its embedding vector $v_i$ is simply the $i$-th row of the embedding matrix $\mathbf{W}_{emb}$.

4.  **Learning Process:** Unlike one-hot encoding, where representations are sparse and orthogonal (meaning no semantic relationship is captured), token embeddings are **learned**. During the training of a neural network (e.g., for language modeling, sentiment analysis, machine translation), the values within the embedding matrix $\mathbf{W}_{emb}$ are updated via backpropagation. The goal is for tokens that are semantically or syntactically similar to have embedding vectors that are close to each other in the vector space (e.g., measured by cosine similarity).

**Example:**

Suppose we have a vocabulary $V = \{\text{cat, dog, walk, run}\}$ and an embedding dimension $d=3$. The embedding matrix might look like this initially (randomly initialized):

$$\mathbf{W}_{emb} = \begin{pmatrix}
0.1 & 0.5 & 0.2 & \text{ (for 'cat', if ID is 0)} \\
0.8 & 0.3 & 0.9 & \text{ (for 'dog', if ID is 1)} \\
0.2 & 0.7 & 0.4 & \text{ (for 'walk', if ID is 2)} \\
0.6 & 0.1 & 0.3 & \text{ (for 'run', if ID is 3)}
\end{pmatrix}$$

If the input token ID is 1 (corresponding to 'dog'), the embedding layer simply performs a lookup and returns the vector $\begin{pmatrix} 0.8 & 0.3 & 0.9 \end{pmatrix}$. As the model trains, these values will adjust to better represent the relationships between 'dog' and other words in its context.

In essence, token embeddings convert sparse, high-dimensional discrete symbols into dense, low-dimensional continuous vectors, capturing rich semantic and syntactic information that is vital for advanced NLP tasks.
```
`
Expectation: The markdown block will explain the mathematical formulation of token embeddings, including the vocabulary, embedding dimension, embedding matrix, and learning process.

**Reasoning**:
Now that the mathematical formulation has been explained, I will proceed with the PyTorch implementation of a token embedding layer as requested in the subtask instructions. This will involve defining a custom PyTorch module that uses `torch.nn.Embedding`.



In [ ]:
import torch
import torch.nn as nn

class TokenEmbedding(nn.Module):
    """
    A PyTorch module for creating token embeddings.

    This module maps discrete token IDs to dense, continuous vector representations.
    It uses `torch.nn.Embedding` internally, which acts as a simple lookup table.
    The embedding vectors are learned during the model's training process.
    """
    def __init__(self, vocab_size: int, embedding_dim: int):
        """
        Initializes the TokenEmbedding layer.

        Args:
            vocab_size (int): The size of the vocabulary (i.e., the total number of unique tokens).
                              This determines the number of rows in the embedding matrix.
            embedding_dim (int): The dimensionality of the embedding vectors.
                                 Each token will be represented by a vector of this size.
                                 This determines the number of columns in the embedding matrix.
        """
        super().__init__()
        # `torch.nn.Embedding` is a simple lookup table that stores fixed-size embeddings for a dictionary of fixed-size vocabularies.
        # The input to the module is a list of indices, and the output is the corresponding word embeddings.
        # The weight matrix (embedding_matrix) has shape (vocab_size, embedding_dim).
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        """
        Performs the forward pass of the TokenEmbedding layer.

        Args:
            token_ids (torch.Tensor): A tensor of token IDs (long integers).
                                      Shape: (batch_size, sequence_length).
                                      Each element in the tensor is an integer representing a token's ID
                                      from the vocabulary.

        Returns:
            torch.Tensor: A tensor of corresponding token embeddings.
                          Shape: (batch_size, sequence_length, embedding_dim).
                          Each token ID is replaced by its 'embedding_dim'-dimensional vector.
        """
        # The `nn.Embedding` layer directly takes the token_ids tensor and performs the lookup.
        # It retrieves the row corresponding to each token ID from its internal embedding matrix.
        return self.embedding(token_ids)

# Example Usage and Verification:
# Define vocabulary size and embedding dimension
example_vocab_size = 10000  # e.g., 10,000 unique words
example_embedding_dim = 256 # e.g., each word is represented by a 256-dimensional vector

# Instantiate the TokenEmbedding layer
embedding_layer = TokenEmbedding(vocab_size=example_vocab_size, embedding_dim=example_embedding_dim)
print(f"TokenEmbedding layer created with vocab_size={example_vocab_size}, embedding_dim={example_embedding_dim}")
print(f"Embedding layer internal weights shape: {embedding_layer.embedding.weight.shape}")

# Create a sample input tensor of token IDs
# Let's say a batch of 2 sentences, with each sentence having 5 tokens
example_token_ids = torch.randint(0, example_vocab_size, (2, 5))
print(f"\nExample input token IDs (shape {example_token_ids.shape}):\n{example_token_ids}")

# Pass the token IDs through the embedding layer
example_embeddings = embedding_layer(example_token_ids)

print(f"\nOutput embeddings shape: {example_embeddings.shape}")
print(f"First token's embedding in the first sequence:\n{example_embeddings[0, 0, :5]}... (showing first 5 dimensions)")

TokenEmbedding layer created with vocab_size=10000, embedding_dim=256
Embedding layer internal weights shape: torch.Size([10000, 256])

Example input token IDs (shape torch.Size([2, 5])):
tensor([[3648, 2450, 9100, 4764, 1943],
        [4651, 4130, 4017, 3715, 5919]])

Output embeddings shape: torch.Size([2, 5, 256])
First token's embedding in the first sequence:
tensor([-0.2634,  1.7858, -0.4407, -0.9067,  1.2316], grad_fn=<SliceBackward0>)... (showing first 5 dimensions)


## Positional Encoding

### Subtask:
Explain the mathematical formulation of sinusoidal positional encoding and implement a PyTorch module to apply it, including detailed comments.


```markdown
## Mathematical Formulation of Sinusoidal Positional Encoding

In sequence-to-sequence models like the Transformer, the self-attention mechanism processes tokens in parallel without an inherent sense of their order. To reintroduce information about the relative or absolute position of tokens in a sequence, positional encodings are added to the token embeddings. Unlike recurrent neural networks (RNNs) that process tokens sequentially, Transformers need an explicit way to incorporate positional information. Sinusoidal positional encoding is a popular choice due to its ability to generalize to unseen sequence lengths.

The core idea is to represent each position in the sequence with a unique vector that is added to its corresponding token embedding. These positional encoding vectors are not learned but are instead generated using specific sine and cosine functions.

For a token at position `pos` (where `pos` ranges from 0 to `max_len - 1`) within an embedding vector of dimension `d_model`, the positional encoding vector $\mathbf{PE}_{pos}$ has `d_model` components. Each component is calculated as follows:

Let $i$ be the dimension index, ranging from $0$ to $d_{model}-1$. The equations for generating the positional encoding values are:

*   For even dimensions ($2i$):
    $$ \text{PE}_{pos, 2i} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right) $$

*   For odd dimensions ($2i+1$):
    $$ \text{PE}_{pos, 2i+1} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right) $$

Where:
-   `pos`: The position of the token in the sequence (e.g., 0, 1, 2, ... `max_len - 1`).
-   `i`: The dimension index within the positional encoding vector. Since we use $2i$ and $2i+1$, `i` effectively ranges from $0$ to $d_{model}/2 - 1$.
-   `d_model`: The dimensionality of the embedding space (and thus the positional encoding vector).
-   $10000^{2i/d_{model}}$: This term creates wavelengths that vary geometrically from $2\pi$ to $10000 \cdot 2\pi$. Using different frequencies for different dimensions allows the model to easily attend to relative positions, as for any fixed offset $k$, $\text{PE}_{pos+k}$ can be represented as a linear function of $\text{PE}_{pos}$.

### How values are generated:

1.  **Frequencies**: For each pair of dimensions ($2i$ and $2i+1$), a unique frequency is determined by the term $1/10000^{2i/d_{model}}$. As `i` increases, this frequency decreases, meaning the sine and cosine waves oscillate slower for higher dimensions.
2.  **Sine and Cosine**: For even dimensions, the sine function is applied to `pos` scaled by its corresponding frequency. For odd dimensions, the cosine function is used. This pairing ensures that each position has a distinct encoding vector and allows for capturing both phase and magnitude information.
3.  **Vector Construction**: These values are concatenated to form the `d_model`-dimensional positional encoding vector for a given `pos`.

### Why this specific encoding is used:

1.  **Injecting Positional Information**: Without positional encoding, the Transformer would treat all tokens as an unordered set. Adding these vectors allows the model to understand the order of words in a sequence.
2.  **Generalization to Longer Sequences**: The sinusoidal functions allow the model to extrapolate to sequence lengths longer than those encountered during training. Any position `pos + k` can be represented as a linear combination of `sin(pos)` and `cos(pos)` because of trigonometric identities ($\sin(A+B) = \sin A \cos B + \cos A \sin B$). This property helps the model learn to attend based on relative positions.
3.  **Distinct Encodings**: Each position generates a unique positional encoding vector, ensuring that no two positions are represented identically.
4.  **Information Density**: Using sine and cosine waves of varying frequencies provides a rich representation where different dimensions can encode different aspects of position, allowing the model to easily distinguish between words at different positions by analyzing the patterns in these vectors.

After generating the positional encoding matrix $\mathbf{PE} \in \mathbb{R}^{\text{max_len} \times d_{model}}$, it is simply added element-wise to the token embeddings before being passed to the subsequent layers of the Transformer.
```

**Reasoning**:
The mathematical formulation of sinusoidal positional encoding has been explained. Now, I will proceed with the PyTorch implementation of a module that applies this encoding, as requested in the subtask instructions. This will involve pre-computing the positional encoding matrix and adding it to the token embeddings.



In [ ]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    """
    A PyTorch module for applying sinusoidal positional encoding to input embeddings.

    This module pre-computes a matrix of positional encodings using sine and cosine functions
    and adds them to the input token embeddings to inject positional information.
    The positional encodings are not learned but are fixed.
    """
    def __init__(self, d_model: int, max_len: int = 5000):
        """
        Initializes the PositionalEncoding layer.

        Args:
            d_model (int): The dimensionality of the embedding space (and thus the positional encoding vectors).
                           This should typically match the `embedding_dim` of the token embeddings.
            max_len (int): The maximum expected sequence length. The positional encoding matrix will be
                           pre-computed up to this length. A larger `max_len` allows for longer sequences
                           but consumes more memory for the pre-computed matrix.
        """
        super().__init__()
        # Create a tensor of shape (max_len, d_model) to store the positional encodings.
        # This matrix will hold PE(pos, i) for each position 'pos' and dimension 'i'.
        pe = torch.zeros(max_len, d_model)

        # Create a tensor for positions (0, 1, ..., max_len-1) of shape (max_len, 1).
        # We need to unsqueeze it to allow broadcasting with 'div_term' later.
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        # Calculate the 'div_term' for the denominator 10000^(2i/d_model).
        # The `i` here represents the index 'k' in the original paper (0, 1, ..., d_model/2 - 1).
        # For even dimensions (2i) and odd dimensions (2i+1), the same 'div_term' is used.
        # exponent is 2i/d_model, so for each 'i' from 0 to d_model/2 - 1, we get an exponent.
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        # Apply sine to even positions in the embedding dimensions.
        # position: (max_len, 1)
        # div_term: (d_model/2)
        # Result of (position * div_term) will be (max_len, d_model/2) due to broadcasting.
        pe[:, 0::2] = torch.sin(position * div_term)

        # Apply cosine to odd positions in the embedding dimensions.
        # pe[:, 1::2] selects all rows, and then columns starting from index 1 with a step of 2 (odd indices).
        pe[:, 1::2] = torch.cos(position * div_term)

        # Register 'pe' as a buffer. Buffers are tensors that are part of the module's state
        # but should not be considered as model parameters (i.e., they are not updated by optimizers).
        # The 'unsqueeze(0)' adds a batch dimension, making 'pe' of shape (1, max_len, d_model).
        # This facilitates element-wise addition with input embeddings of shape (batch_size, sequence_length, d_model).
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Adds the positional encodings to the input embeddings.

        Args:
            x (torch.Tensor): The input tensor of token embeddings.
                              Shape: (batch_size, sequence_length, d_model).

        Returns:
            torch.Tensor: The input embeddings with positional encodings added.
                          Shape: (batch_size, sequence_length, d_model).
        """
        # Add the pre-computed positional encodings to the input tensor.
        # We slice the 'pe' buffer to match the sequence length of the input 'x'.
        # The 'pe' buffer has a batch dimension of 1, so it broadcasts correctly across the batch.
        # x is (batch_size, sequence_length, d_model)
        # self.pe is (1, max_len, d_model)
        # self.pe[:, :x.size(1)] slices to (1, sequence_length, d_model)
        # The addition will broadcast across the batch dimension.
        return x + self.pe[:, :x.size(1)]

# Example Usage and Verification:
# Define model dimension and max sequence length
example_d_model = 256
example_max_len = 500

# Instantiate the PositionalEncoding layer
pos_encoder = PositionalEncoding(d_model=example_d_model, max_len=example_max_len)
print(f"PositionalEncoding layer created with d_model={example_d_model}, max_len={example_max_len}")
print(f"Pre-computed PE matrix shape: {pos_encoder.pe.shape}")

# Create a sample input tensor (e.g., from TokenEmbedding output)
# Let's say a batch of 2 sequences, each with 10 tokens (sequence_length=10)
example_batch_size = 2
example_sequence_length = 10
example_token_embeddings = torch.randn(example_batch_size, example_sequence_length, example_d_model)
print(f"\nExample input token embeddings shape: {example_token_embeddings.shape}")

# Pass the embeddings through the positional encoding layer
example_embeddings_with_pos = pos_encoder(example_token_embeddings)

print(f"Output embeddings with PE shape: {example_embeddings_with_pos.shape}")
print(f"Original embedding for first token in first sequence (first 5 dims):\n{example_token_embeddings[0, 0, :5]}")
print(f"Positional encoding for first token (pos=0, first 5 dims):\n{pos_encoder.pe[0, 0, :5]}")
print(f"Combined embedding for first token (pos=0, first 5 dims):\n{example_embeddings_with_pos[0, 0, :5]}")

# Verify that PE is added: (should be roughly equal)
# Combined = Original + PE
# Original = Combined - PE
print(f"\nVerifying addition (Original vs Combined - PE):\n{example_token_embeddings[0, 0, :5]}\n{example_embeddings_with_pos[0, 0, :5] - pos_encoder.pe[0, 0, :5]}")
assert torch.allclose(example_token_embeddings[0, 0], example_embeddings_with_pos[0, 0] - pos_encoder.pe[0, 0])
print("Verification successful: Positional encodings were correctly added.")

PositionalEncoding layer created with d_model=256, max_len=500
Pre-computed PE matrix shape: torch.Size([1, 500, 256])

Example input token embeddings shape: torch.Size([2, 10, 256])
Output embeddings with PE shape: torch.Size([2, 10, 256])
Original embedding for first token in first sequence (first 5 dims):
tensor([ 0.1309,  0.8094, -0.6432,  1.5950, -0.3493])
Positional encoding for first token (pos=0, first 5 dims):
tensor([0., 1., 0., 1., 0.])
Combined embedding for first token (pos=0, first 5 dims):
tensor([ 0.1309,  1.8094, -0.6432,  2.5950, -0.3493])

Verifying addition (Original vs Combined - PE):
tensor([ 0.1309,  0.8094, -0.6432,  1.5950, -0.3493])
tensor([ 0.1309,  0.8094, -0.6432,  1.5950, -0.3493])
Verification successful: Positional encodings were correctly added.


## Multi-Head Self-Attention

### Subtask:
Explain the mathematical formulation of self-attention and multi-head attention.


```markdown
## Mathematical Formulation of Self-Attention and Multi-Head Attention

### 1. Self-Attention

Self-attention is a mechanism that allows a model to weigh the importance of different words (or tokens) in an input sequence when encoding a particular word. It computes a "context vector" for each word by aggregating information from all other words in the sequence, based on how relevant they are to the current word.

Let the input sequence of token embeddings be represented by a matrix $\mathbf{X} \in \mathbb{R}^{L \times d_{model}}$, where $L$ is the sequence length and $d_{model}$ is the dimensionality of each embedding.

The self-attention mechanism involves three key components for each token: a Query (Q), a Key (K), and a Value (V).

1.  **Linear Projections to Query, Key, and Value**: The input embeddings $\mathbf{X}$ are first linearly transformed into three different matrices: Query ($\mathbf{Q}$), Key ($\mathbf{K}$), and Value ($\mathbf{V}$). These transformations are done using learned weight matrices:

    *   $\mathbf{Q} = \mathbf{X} \mathbf{W}^Q \in \mathbb{R}^{L \times d_k}$
    *   $\mathbf{K} = \mathbf{X} \mathbf{W}^K \in \mathbb{R}^{L \times d_k}$
    *   $\mathbf{V} = \mathbf{X} \mathbf{W}^V \in \mathbb{R}^{L \times d_v}$

    Where:
    *   $\mathbf{W}^Q \in \mathbb{R}^{d_{model} \times d_k}$ is the weight matrix for Queries.
    *   $\mathbf{W}^K \in \mathbb{R}^{d_{model} \times d_k}$ is the weight matrix for Keys.
    *   $\mathbf{W}^V \in \mathbb{R}^{d_{model} \times d_v}$ is the weight matrix for Values.
    *   $d_k$ is the dimension of Queries and Keys (typically $d_k = d_{model} / h$, where $h$ is the number of heads).
    *   $d_v$ is the dimension of Values (typically $d_v = d_{model} / h$).

2.  **Scaled Dot-Product Attention**: This is the core calculation for determining attention weights.

    First, we calculate the dot product between the Query and Key matrices: $\mathbf{Q} \mathbf{K}^T$. This produces a score matrix that indicates the similarity between each query (token) and all keys (other tokens).

    $$ \text{Scores} = \mathbf{Q} \mathbf{K}^T \in \mathbb{R}^{L \times L} $$

    Next, these scores are scaled by $\sqrt{d_k}$. This scaling is important to prevent the dot products from becoming too large, which can push the softmax function into regions with extremely small gradients, leading to training instability.

    $$ \text{Scaled Scores} = \frac{\mathbf{Q} \mathbf{K}^T}{\sqrt{d_k}} $$

    Optionally, a mask (e.g., for padding or for masked self-attention in a decoder) can be applied to the scaled scores. For decoder self-attention, a look-ahead mask is used to prevent positions from attending to subsequent positions. This involves setting values for future positions to $-\infty$ before the softmax.

    $$ \text{Masked Scaled Scores} = \frac{\mathbf{Q} \mathbf{K}^T}{\sqrt{d_k}} + \mathbf{M} $$

    Where $\mathbf{M}$ is the mask matrix, with $-\infty$ for masked positions and $0$ otherwise.

    Then, the softmax function is applied row-wise to get the attention weights. This normalizes the scores so that they sum to 1 for each query, representing the probability distribution over the values.

    $$ \text{Attention Weights} = \text{softmax}\left(\frac{\mathbf{Q} \mathbf{K}^T}{\sqrt{d_k}} + \mathbf{M}\right) \in \mathbb{R}^{L \times L} $$

3.  **Output Calculation**: Finally, these attention weights are multiplied by the Value matrix $\mathbf{V}$. This is essentially a weighted sum of the value vectors, where the weights are determined by the attention scores.

    $$ \text{Self-Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\left(\frac{\mathbf{Q} \mathbf{K}^T}{\sqrt{d_k}} + \mathbf{M}\right) \mathbf{V} \in \mathbb{R}^{L \times d_v} $$

This output matrix contains the context-aware representation for each token in the sequence.

### 2. Multi-Head Attention

Multi-head attention extends the self-attention mechanism by allowing the model to jointly attend to information from different representation subspaces at different positions. Instead of performing a single attention function with $d_{model}$-dimensional Keys, Values, and Queries, we linearly project the Queries, Keys, and Values $h$ times with different, learned linear projections. This allows each 'head' to focus on different aspects of the input sequence.

1.  **Multiple Projections**: The input embeddings $\mathbf{X}$ are projected $h$ times into Query, Key, and Value matrices, each with smaller dimensions $d_k$ and $d_v$. For each head $j \in \{1, \dots, h\}$:

    *   $\mathbf{Q}_j = \mathbf{X} \mathbf{W}^Q_j \in \mathbb{R}^{L \times d_k}$
    *   $\mathbf{K}_j = \mathbf{X} \mathbf{W}^K_j \in \mathbb{R}^{L \times d_k}$
    *   $\mathbf{V}_j = \mathbf{X} \mathbf{W}^V_j \in \mathbb{R}^{L \times d_v}$

    Where:
    *   $\mathbf{W}^Q_j \in \mathbb{R}^{d_{model} \times d_k}$, $\mathbf{W}^K_j \in \mathbb{R}^{d_{model} \times d_k}$, $\mathbf{W}^V_j \in \mathbb{R}^{d_{model} \times d_v}$ are distinct, learned weight matrices for each head $j$.
    *   Typically, $d_k = d_v = d_{model} / h$. This ensures that the concatenated output dimension matches the input $d_{model}$.

2.  **Parallel Attention Execution**: Each of these projected Query, Key, and Value sets goes through its own scaled dot-product attention calculation independently:

    $$ \text{head}_j = \text{Self-Attention}(\mathbf{Q}_j, \mathbf{K}_j, \mathbf{V}_j) = \text{softmax}\left(\frac{\mathbf{Q}_j \mathbf{K}_j^T}{\sqrt{d_k}} + \mathbf{M}\right) \mathbf{V}_j \in \mathbb{R}^{L \times d_v} $$

3.  **Concatenation**: The outputs from all $h$ attention heads are then concatenated along the dimension of $d_v$ to form a single matrix. Since each head produces an output of shape $(L \times d_v)$, concatenating $h$ such outputs results in a matrix of shape $(L \times h \cdot d_v)$. Given $h \cdot d_v = h \cdot (d_{model} / h) = d_{model}$, the concatenated output has the same dimension as the original input embeddings.

    $$ \text{Concat}(\text{head}_1, \dots, \text{head}_h) \in \mathbb{R}^{L \times (h \cdot d_v)} $$

4.  **Final Linear Transformation**: The concatenated output is then passed through a final linear projection, using a learned weight matrix $\mathbf{W}^O$, to combine the information from all heads and transform it back to the original $d_{model}$ dimensionality.

    $$ \text{MultiHead}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) \mathbf{W}^O \in \mathbb{R}^{L \times d_{model}} $$

    Where $\mathbf{W}^O \in \mathbb{R}^{(h \cdot d_v) \times d_{model}}$ is the learned output weight matrix.

This multi-head mechanism allows the model to learn various types of relationships and dependencies (e.g., syntactic, semantic, long-range, short-range) simultaneously within the sequence, enhancing its representational power and robustness.
```
```


## Multi-Head Self-Attention

### Subtask:
Implement a PyTorch module for multi-head self-attention, including detailed comments.


```markdown
## Mathematical Formulation of Multi-Head Self-Attention

Multi-Head Self-Attention (MHSA) is a core component of the Transformer architecture, designed to allow the model to jointly attend to information from different representation subspaces at different positions. It is an extension of the basic scaled dot-product attention mechanism.

### 1. Scaled Dot-Product Attention

The fundamental unit of attention is the scaled dot-product attention. Given a Query (Q), Key (K), and Value (V) matrices, the output is calculated as:

$$ \text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V $$

Where:
-   $Q \in \mathbb{R}^{n \times d_k}$ is the Query matrix, representing the current token(s) we are focusing on.
-   $K \in \mathbb{R}^{m \times d_k}$ is the Key matrix, representing all available tokens that might be relevant.
-   $V \in \mathbb{R}^{m \times d_v}$ is the Value matrix, containing the actual information to be extracted from the relevant tokens.
-   $n$ is the sequence length of the queries, and $m$ is the sequence length of the keys/values.
-   $d_k$ is the dimension of the keys and queries.
-   $d_v$ is the dimension of the values.
-   $\frac{1}{\sqrt{d_k}}$ is a scaling factor to prevent the dot products from becoming too large, which can push the softmax function into regions with extremely small gradients, hindering learning.

### 2. Multi-Head Mechanism

Instead of performing a single attention function with $d_{model}$-dimensional keys, values, and queries, Multi-Head Attention linearly projects the queries, keys, and values $h$ times with different, learned linear projections to $d_k$, $d_k$, and $d_v$ dimensions, respectively. These projections are applied independently for each head.

First, for each head $j \in \{1, \dots, h\}$:

$$ Q_j = QW_j^Q \quad K_j = KW_j^K \quad V_j = VW_j^V $$

Where:
-   $W_j^Q \in \mathbb{R}^{d_{model} \times d_k}$, $W_j^K \in \mathbb{R}^{d_{model} \times d_k}$, $W_j^V \in \mathbb{R}^{d_{model} \times d_v}$ are the learnable projection matrices for head $j$.
-   Typically, $d_k = d_v = d_{model} / h$, where $h$ is the number of heads. This ensures that the total computational cost and parameter count remain similar to single-head attention with full $d_{model}$ dimensions.

Then, the scaled dot-product attention is computed for each head $j$ independently:

$$ \text{head}_j = \text{Attention}(Q_j, K_j, V_j) $$

### 3. Concatenation and Final Linear Projection

The outputs from all $h$ attention heads are then concatenated and linearly transformed back into a $d_{model}$-dimensional output. This allows the model to combine information from all different representation subspaces.

$$ \text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h)W^O $$

Where:
-   $W^O \in \mathbb{R}^{h \cdot d_v \times d_{model}}$ is the final learnable linear projection matrix.

### Why Multi-Head Attention?

1.  **Enriched Representation**: By allowing the model to jointly attend to information from different representation subspaces at different positions, MHSA enhances the model's ability to focus on different aspects of the input sequence simultaneously.
2.  **Diverse Focus**: Each head learns to attend to different parts of the input or to different types of relationships (e.g., one head might focus on syntactic dependencies, another on semantic relationships).
3.  **Increased Model Capacity**: The multiple projection matrices provide more parameters, increasing the model's capacity to learn complex patterns.
4.  **Parallel Computation**: The attention calculations for each head can be performed in parallel, which is computationally efficient.

In summary, Multi-Head Self-Attention effectively allows the model to perform multiple "looks" at the input sequence from different perspectives, combining these perspectives to form a richer and more comprehensive understanding.

**Reasoning**:
The mathematical formulation for Multi-Head Self-Attention has been explained in the previous step. Now, I will proceed with the PyTorch implementation of the `MultiHeadSelfAttention` module, adhering to the detailed instructions provided in the subtask.



In [ ]:
import torch
import torch.nn as nn
import math

class MultiHeadSelfAttention(nn.Module):
    """
    A PyTorch module for Multi-Head Self-Attention (MHSA).

    MHSA allows the model to jointly attend to information from different
    representation subspaces at different positions. It uses multiple
    "heads" to perform attention in parallel, and then concatenates their
    results and projects them back.
    """
    def __init__(self, d_model: int, num_heads: int):
        """
        Initializes the MultiHeadSelfAttention layer.

        Args:
            d_model (int): The dimensionality of the input and output embeddings.
                           This is also the dimension of the model's internal representation.
            num_heads (int): The number of attention heads. `d_model` must be divisible by `num_heads`.
        """
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError(f"d_model ({d_model}) must be divisible by num_heads ({num_heads})")

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads # Dimension of each head's query, key, and value vectors

        # Linear layers for Query, Key, Value projections
        # Each projects the input from d_model to d_model, which then gets split into num_heads * head_dim
        self.query_proj = nn.Linear(d_model, d_model)
        self.key_proj = nn.Linear(d_model, d_model)
        self.value_proj = nn.Linear(d_model, d_model)

        # Final linear layer to project the concatenated output of all heads back to d_model
        self.output_proj = nn.Linear(d_model, d_model)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        """
        Performs the forward pass of the MultiHeadSelfAttention layer.

        Args:
            x (torch.Tensor): The input tensor of shape (batch_size, sequence_length, d_model).
                              This typically comes from token embeddings + positional encodings.
            mask (torch.Tensor, optional): An optional mask tensor, typically used for padded
                                           elements or for masked self-attention in decoders.
                                           Shape: (batch_size, 1, sequence_length, sequence_length)
                                           or (batch_size, sequence_length, sequence_length).
                                           Elements where mask is True (or 0 depending on convention)
                                           are ignored/masked out in attention calculation.
                                           Here, we assume mask=0 means ignore.

        Returns:
            torch.Tensor: The output tensor of shape (batch_size, sequence_length, d_model),
                          after applying multi-head self-attention and final projection.
        """
        batch_size, sequence_length, _ = x.size()

        # 1. Apply linear projections for Query, Key, Value
        # Each projection transforms 'x' from (batch_size, seq_len, d_model)
        # to (batch_size, seq_len, d_model)
        Q = self.query_proj(x) # (batch_size, seq_len, d_model)
        K = self.key_proj(x)   # (batch_size, seq_len, d_model)
        V = self.value_proj(x) # (batch_size, seq_len, d_model)

        # 2. Reshape and Transpose for Multi-Head Attention
        # Split d_model into num_heads * head_dim and rearrange dimensions
        # Reshape: (batch_size, seq_len, d_model) -> (batch_size, seq_len, num_heads, head_dim)
        # Transpose: (batch_size, seq_len, num_heads, head_dim) -> (batch_size, num_heads, seq_len, head_dim)
        # This makes each head's data contiguous and ready for batch matrix multiplication.
        Q = Q.view(batch_size, sequence_length, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, sequence_length, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, sequence_length, self.num_heads, self.head_dim).transpose(1, 2)
        # Now Q, K, V are all of shape (batch_size, num_heads, seq_len, head_dim)

        # 3. Compute Attention Scores (Scaled Dot-Product Attention)
        # Matmul Q and K_transpose: (..., seq_len, head_dim) @ (..., head_dim, seq_len)
        # Resulting scores shape: (batch_size, num_heads, seq_len, seq_len)
        attention_scores = torch.matmul(Q, K.transpose(-2, -1)) # K.transpose(-2, -1) swaps the last two dimensions of K

        # Scale the attention scores by 1/sqrt(d_k) to prevent large values from saturating softmax
        attention_scores = attention_scores / math.sqrt(self.head_dim)

        # 4. Apply Mask (if provided)
        # Typically, mask values of 0 are used to indicate elements to ignore (e.g., padding).
        # Masking is done by filling the positions to be ignored with a very small negative number
        # so that their softmax probabilities become close to zero.
        if mask is not None:
            # The mask should be broadcastable. Common shapes: (batch_size, 1, seq_len, seq_len)
            # or (1, 1, seq_len, seq_len) for decoder masking.
            attention_scores = attention_scores.masked_fill(mask == 0, float('-inf'))

        # 5. Apply Softmax to get Attention Weights
        # Softmax is applied along the last dimension (over the keys for each query)
        attention_weights = torch.softmax(attention_scores, dim=-1)
        # attention_weights shape: (batch_size, num_heads, seq_len, seq_len)

        # 6. Compute Weighted Sum of Values
        # Matmul attention_weights and V: (..., seq_len, seq_len) @ (..., seq_len, head_dim)
        # Resulting output of each head shape: (batch_size, num_heads, seq_len, head_dim)
        output = torch.matmul(attention_weights, V)

        # 7. Concatenate and Apply Final Linear Projection
        # Transpose back: (batch_size, num_heads, seq_len, head_dim) -> (batch_size, seq_len, num_heads, head_dim)
        # Reshape (concatenate): (batch_size, seq_len, num_heads, head_dim) -> (batch_size, seq_len, d_model)
        # This effectively concatenates the outputs of all heads.
        output = output.transpose(1, 2).contiguous().view(batch_size, sequence_length, self.d_model)

        # Apply the final linear projection
        output = self.output_proj(output)
        # output shape: (batch_size, seq_len, d_model)

        return output

# Example Usage and Verification:
# Define parameters
example_d_model = 256
example_num_heads = 8
example_batch_size = 2
example_sequence_length = 10

# Instantiate MultiHeadSelfAttention layer
attention_layer = MultiHeadSelfAttention(d_model=example_d_model, num_heads=example_num_heads)
print(f"MultiHeadSelfAttention layer created with d_model={example_d_model}, num_heads={example_num_heads}")
print(f"Each head dimension (head_dim): {attention_layer.head_dim}")

# Create a sample input tensor (e.g., from TokenEmbedding + PositionalEncoding)
example_input_tensor = torch.randn(example_batch_size, example_sequence_length, example_d_model)
print(f"\nExample input tensor shape: {example_input_tensor.shape}")

# Forward pass without mask
output_no_mask = attention_layer(example_input_tensor)
print(f"Output shape (no mask): {output_no_mask.shape}")

# Create a sample mask (e.g., a causal mask for decoder self-attention)
# For a sequence of length L, a causal mask is a lower triangular matrix
# where the diagonal and upper triangle are 0 (or -inf for softmax masking)
# and the lower triangle is 1 (or 0 for softmax masking, if mask=0 means ignore).
# Here, we'll create a mask where future positions are masked out.
causal_mask = torch.tril(torch.ones(example_sequence_length, example_sequence_length)).bool()
# Expand mask for broadcasting across batch and heads: (1, 1, seq_len, seq_len)
causal_mask = causal_mask.unsqueeze(0).unsqueeze(0)
print(f"\nExample causal mask shape: {causal_mask.shape}")
print(f"""Example causal mask (first 5x5):
{causal_mask[0,0,:5,:5]}""")

# Forward pass with mask
output_with_mask = attention_layer(example_input_tensor, mask=causal_mask)
print(f"Output shape (with mask): {output_with_mask.shape}")

# Verify output dimensions match expected
assert output_no_mask.shape == (example_batch_size, example_sequence_length, example_d_model)
assert output_with_mask.shape == (example_batch_size, example_sequence_length, example_d_model)
print("Verification successful: Output shapes are correct.")


SyntaxError: unterminated f-string literal (detected at line 146) (ipython-input-2609605579.py, line 146)

**Reasoning**:
The previous code failed due to a `SyntaxError: unterminated f-string literal`. This happened because the f-string for printing the causal mask was split across multiple lines without using triple quotes. I will correct the f-string to use triple quotes, allowing it to span multiple lines correctly.



In [ ]:
import torch
import torch.nn as nn
import math

class MultiHeadSelfAttention(nn.Module):
    """
    A PyTorch module for Multi-Head Self-Attention (MHSA).

    MHSA allows the model to jointly attend to information from different
    representation subspaces at different positions. It uses multiple
    "heads" to perform attention in parallel, and then concatenates their
    results and projects them back.
    """
    def __init__(self, d_model: int, num_heads: int):
        """
        Initializes the MultiHeadSelfAttention layer.

        Args:
            d_model (int): The dimensionality of the input and output embeddings.
                           This is also the dimension of the model's internal representation.
            num_heads (int): The number of attention heads. `d_model` must be divisible by `num_heads`.
        """
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError(f"d_model ({d_model}) must be divisible by num_heads ({num_heads})")

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads # Dimension of each head's query, key, and value vectors

        # Linear layers for Query, Key, Value projections
        # Each projects the input from d_model to d_model, which then gets split into num_heads * head_dim
        self.query_proj = nn.Linear(d_model, d_model)
        self.key_proj = nn.Linear(d_model, d_model)
        self.value_proj = nn.Linear(d_model, d_model)

        # Final linear layer to project the concatenated output of all heads back to d_model
        self.output_proj = nn.Linear(d_model, d_model)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        """
        Performs the forward pass of the MultiHeadSelfAttention layer.

        Args:
            x (torch.Tensor): The input tensor of shape (batch_size, sequence_length, d_model).
                              This typically comes from token embeddings + positional encodings.
            mask (torch.Tensor, optional): An optional mask tensor, typically used for padded
                                           elements or for masked self-attention in decoders.
                                           Shape: (batch_size, 1, sequence_length, sequence_length)
                                           or (batch_size, sequence_length, sequence_length).
                                           Elements where mask is True (or 0 depending on convention)
                                           are ignored/masked out in attention calculation.
                                           Here, we assume mask=0 means ignore.

        Returns:
            torch.Tensor: The output tensor of shape (batch_size, sequence_length, d_model),
                          after applying multi-head self-attention and final projection.
        """
        batch_size, sequence_length, _ = x.size()

        # 1. Apply linear projections for Query, Key, Value
        # Each projection transforms 'x' from (batch_size, seq_len, d_model)
        # to (batch_size, seq_len, d_model)
        Q = self.query_proj(x) # (batch_size, seq_len, d_model)
        K = self.key_proj(x)   # (batch_size, seq_len, d_model)
        V = self.value_proj(x) # (batch_size, seq_len, d_model)

        # 2. Reshape and Transpose for Multi-Head Attention
        # Split d_model into num_heads * head_dim and rearrange dimensions
        # Reshape: (batch_size, seq_len, d_model) -> (batch_size, seq_len, num_heads, head_dim)
        # Transpose: (batch_size, seq_len, num_heads, head_dim) -> (batch_size, num_heads, seq_len, head_dim)
        # This makes each head's data contiguous and ready for batch matrix multiplication.
        Q = Q.view(batch_size, sequence_length, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, sequence_length, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, sequence_length, self.num_heads, self.head_dim).transpose(1, 2)
        # Now Q, K, V are all of shape (batch_size, num_heads, seq_len, head_dim)

        # 3. Compute Attention Scores (Scaled Dot-Product Attention)
        # Matmul Q and K_transpose: (..., seq_len, head_dim) @ (..., head_dim, seq_len)
        # Resulting scores shape: (batch_size, num_heads, seq_len, seq_len)
        attention_scores = torch.matmul(Q, K.transpose(-2, -1)) # K.transpose(-2, -1) swaps the last two dimensions of K

        # Scale the attention scores by 1/sqrt(d_k) to prevent large values from saturating softmax
        attention_scores = attention_scores / math.sqrt(self.head_dim)

        # 4. Apply Mask (if provided)
        # Typically, mask values of 0 are used to indicate elements to ignore (e.g., padding).
        # Masking is done by filling the positions to be ignored with a very small negative number
        # so that their softmax probabilities become close to zero.
        if mask is not None:
            # The mask should be broadcastable. Common shapes: (batch_size, 1, seq_len, seq_len)
            # or (1, 1, seq_len, seq_len) for decoder masking.
            attention_scores = attention_scores.masked_fill(mask == 0, float('-inf'))

        # 5. Apply Softmax to get Attention Weights
        # Softmax is applied along the last dimension (over the keys for each query)
        attention_weights = torch.softmax(attention_scores, dim=-1)
        # attention_weights shape: (batch_size, num_heads, seq_len, seq_len)

        # 6. Compute Weighted Sum of Values
        # Matmul attention_weights and V: (..., seq_len, seq_len) @ (..., seq_len, head_dim)
        # Resulting output of each head shape: (batch_size, num_heads, seq_len, head_dim)
        output = torch.matmul(attention_weights, V)

        # 7. Concatenate and Apply Final Linear Projection
        # Transpose back: (batch_size, num_heads, seq_len, head_dim) -> (batch_size, seq_len, num_heads, head_dim)
        # Reshape (concatenate): (batch_size, seq_len, num_heads, head_dim) -> (batch_size, seq_len, d_model)
        # This effectively concatenates the outputs of all heads.
        output = output.transpose(1, 2).contiguous().view(batch_size, sequence_length, self.d_model)

        # Apply the final linear projection
        output = self.output_proj(output)
        # output shape: (batch_size, seq_len, d_model)

        return output

# Example Usage and Verification:
# Define parameters
example_d_model = 256
example_num_heads = 8
example_batch_size = 2
example_sequence_length = 10

# Instantiate MultiHeadSelfAttention layer
attention_layer = MultiHeadSelfAttention(d_model=example_d_model, num_heads=example_num_heads)
print(f"MultiHeadSelfAttention layer created with d_model={example_d_model}, num_heads={example_num_heads}")
print(f"Each head dimension (head_dim): {attention_layer.head_dim}")

# Create a sample input tensor (e.g., from TokenEmbedding + PositionalEncoding)
example_input_tensor = torch.randn(example_batch_size, example_sequence_length, example_d_model)
print(f"\nExample input tensor shape: {example_input_tensor.shape}")

# Forward pass without mask
output_no_mask = attention_layer(example_input_tensor)
print(f"Output shape (no mask): {output_no_mask.shape}")

# Create a sample mask (e.g., a causal mask for decoder self-attention)
# For a sequence of length L, a causal mask is a lower triangular matrix
# where the diagonal and upper triangle are 0 (or -inf for softmax masking)
# and the lower triangle is 1 (or 0 for softmax masking, if mask=0 means ignore).
# Here, we'll create a mask where future positions are masked out.
causal_mask = torch.tril(torch.ones(example_sequence_length, example_sequence_length)).bool()
# Expand mask for broadcasting across batch and heads: (1, 1, seq_len, seq_len)
causal_mask = causal_mask.unsqueeze(0).unsqueeze(0)
print(f"\nExample causal mask shape: {causal_mask.shape}")
print(f"""Example causal mask (first 5x5):
{causal_mask[0,0,:5,:5]}""")

# Forward pass with mask
output_with_mask = attention_layer(example_input_tensor, mask=causal_mask)
print(f"Output shape (with mask): {output_with_mask.shape}")

# Verify output dimensions match expected
assert output_no_mask.shape == (example_batch_size, example_sequence_length, example_d_model)
assert output_with_mask.shape == (example_batch_size, example_sequence_length, example_d_model)
print("Verification successful: Output shapes are correct.")

MultiHeadSelfAttention layer created with d_model=256, num_heads=8
Each head dimension (head_dim): 32

Example input tensor shape: torch.Size([2, 10, 256])
Output shape (no mask): torch.Size([2, 10, 256])

Example causal mask shape: torch.Size([1, 1, 10, 10])
Example causal mask (first 5x5):
tensor([[ True, False, False, False, False],
        [ True,  True, False, False, False],
        [ True,  True,  True, False, False],
        [ True,  True,  True,  True, False],
        [ True,  True,  True,  True,  True]])
Output shape (with mask): torch.Size([2, 10, 256])
Verification successful: Output shapes are correct.


## Masked Attention

### Subtask:
Explain the purpose and mathematical application of masked attention in a decoder. Integrate masking functionality into the previously implemented attention module, including detailed comments.


```markdown
## Masked Attention in Decoders

Masked attention is a crucial mechanism primarily used in the decoder component of sequence-to-sequence models, such as the Transformer. Its main purpose is to enforce **causality** and prevent the decoder from 'cheating' by looking at future tokens in the target sequence during training.

### 1. Purpose of Masked Attention

In a decoder, especially when generating a sequence token by token (e.g., in language generation or machine translation), the prediction of the current token should only depend on the previously generated tokens. If the decoder were allowed to attend to all tokens in the target sequence, including those that come after the current position, it would have access to the 'answer' and the learning process would be trivialized. This is particularly important during the training phase where the full target sequence is available.

**Key Role:** To ensure that for a given output position `t`, the attention mechanism can only consider information from positions `1` to `t` (inclusive) and not from positions `t+1` onwards.

### 2. Mathematical Application of Masking

The masking operation is applied directly to the attention scores (the output of $\mathbf{Q} \mathbf{K}^T$) *before* the softmax function. This is critical because modifying the scores before softmax ensures that these 'future' positions receive a probability of virtually zero after softmax, effectively preventing them from contributing to the weighted sum of values.

The scaled dot-product attention formula, with masking, is:

$$ \text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\left(\frac{\mathbf{Q} \mathbf{K}^T}{\sqrt{d_k}} + \mathbf{M}\right)\mathbf{V} $$

Where:
-   $\mathbf{Q} \mathbf{K}^T / \sqrt{d_k}$ are the scaled attention scores.
-   $\mathbf{M}$ is the **mask matrix**. For positions $(i, j)$ where $j > i$ (i.e., token $j$ is a future token relative to token $i$), the corresponding element $M_{ij}$ is set to $-\infty$. For all other positions ($j \le i$), $M_{ij}$ is typically set to 0. (In practice, a boolean mask might be used where `True` indicates values to keep and `False` indicates values to mask out, which then get filled with $-\infty$).

When $-\infty$ is added to the attention scores for future positions, applying the `softmax` function results in $e^{-\infty}$ which is 0. This makes the attention weights for future tokens zero, meaning they have no influence on the output for the current token.

### 3. Integration into `MultiHeadSelfAttention` Module

The `MultiHeadSelfAttention` module implemented in the previous step already includes the necessary functionality to apply masking. This is handled by the `mask` parameter in the `forward` method.

Specifically, the following lines of code within the `forward` method are responsible for applying the mask:

```python
        # ... (previous code)

        # 4. Apply Mask (if provided)
        # Typically, mask values of 0 are used to indicate elements to ignore (e.g., padding).
        # Masking is done by filling the positions to be ignored with a very small negative number
        # so that their softmax probabilities become close to zero.
        if mask is not None:
            # The mask should be broadcastable. Common shapes: (batch_size, 1, seq_len, seq_len)
            # or (1, 1, seq_len, seq_len) for decoder masking.
            attention_scores = attention_scores.masked_fill(mask == 0, float('-inf'))

        # 5. Apply Softmax to get Attention Weights
        attention_weights = torch.softmax(attention_scores, dim=-1)

        # ... (subsequent code)
```

In this implementation:
-   The `mask` parameter is a `torch.Tensor`.
-   The condition `mask == 0` is used to identify the positions that should be masked (e.g., `False` in a boolean mask, or `0` in a binary mask if we follow the convention that `0` means 'ignore').
-   `attention_scores.masked_fill(mask == 0, float('-inf'))` performs the mathematical operation described: it replaces the scaled attention scores at masked positions with a very small negative number (`-inf`), ensuring their softmax probability becomes zero.

### 4. Review of Comments

The comments within the `MultiHeadSelfAttention` class accurately describe the masking mechanism, explaining that `mask == 0` indicates elements to ignore and that `masked_fill` is used to set these scores to `float('-inf')` before `softmax` to effectively nullify their contribution.

Therefore, the existing `MultiHeadSelfAttention` module is already capable of performing masked attention when a suitable mask (like a causal/look-ahead mask) is provided to its `forward` method.

```markdown
## Masked Attention in Decoders

Masked attention is a crucial mechanism primarily used in the decoder component of sequence-to-sequence models, such as the Transformer. Its main purpose is to enforce **causality** and prevent the decoder from 'cheating' by looking at future tokens in the target sequence during training.

### 1. Purpose of Masked Attention

In a decoder, especially when generating a sequence token by token (e.g., in language generation or machine translation), the prediction of the current token should only depend on the previously generated tokens. If the decoder were allowed to attend to all tokens in the target sequence, including those that come after the current position, it would have access to the 'answer' and the learning process would be trivialized. This is particularly important during the training phase where the full target sequence is available.

**Key Role:** To ensure that for a given output position `t`, the attention mechanism can only consider information from positions `1` to `t` (inclusive) and not from positions `t+1` onwards.

### 2. Mathematical Application of Masking

The masking operation is applied directly to the attention scores (the output of $\mathbf{Q} \mathbf{K}^T$) *before* the softmax function. This is critical because modifying the scores before softmax ensures that these 'future' positions receive a probability of virtually zero after softmax, effectively preventing them from contributing to the weighted sum of values.

The scaled dot-product attention formula, with masking, is:

$$ \text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\left(\frac{\mathbf{Q} \mathbf{K}^T}{\sqrt{d_k}} + \mathbf{M}\right)\mathbf{V} $$

Where:
-   $\mathbf{Q} \mathbf{K}^T / \sqrt{d_k}$ are the scaled attention scores.
-   $\mathbf{M}$ is the **mask matrix**. For positions $(i, j)$ where $j > i$ (i.e., token $j$ is a future token relative to token $i$), the corresponding element $M_{ij}$ is set to $-\infty$. For all other positions ($j \le i$), $M_{ij}$ is typically set to 0. (In practice, a boolean mask might be used where `True` indicates values to keep and `False` indicates values to mask out, which then get filled with $-\infty$).

When $-\infty$ is added to the attention scores for future positions, applying the `softmax` function results in $e^{-\infty}$ which is 0. This makes the attention weights for future tokens zero, meaning they have no influence on the output for the current token.

### 3. Integration into `MultiHeadSelfAttention` Module

The `MultiHeadSelfAttention` module implemented in the previous step already includes the necessary functionality to apply masking. This is handled by the `mask` parameter in the `forward` method.

Specifically, the following lines of code within the `forward` method are responsible for applying the mask:

```python
        # ... (previous code)

        # 4. Apply Mask (if provided)
        # Typically, mask values of 0 are used to indicate elements to ignore (e.g., padding).
        # Masking is done by filling the positions to be ignored with a very small negative number
        # so that their softmax probabilities become close to zero.
        if mask is not None:
            # The mask should be broadcastable. Common shapes: (batch_size, 1, seq_len, seq_len)
            # or (1, 1, seq_len, seq_len) for decoder masking.
            attention_scores = attention_scores.masked_fill(mask == 0, float('-inf'))

        # 5. Apply Softmax to get Attention Weights
        attention_weights = torch.softmax(attention_scores, dim=-1)

        # ... (subsequent code)
```

In this implementation:
-   The `mask` parameter is a `torch.Tensor`.
-   The condition `mask == 0` is used to identify the positions that should be masked (e.g., `False` in a boolean mask, or `0` in a binary mask if we follow the convention that `0` means 'ignore').
-   `attention_scores.masked_fill(mask == 0, float('-inf'))` performs the mathematical operation described: it replaces the scaled attention scores at masked positions with a very small negative number (`-inf`), ensuring their softmax probability becomes zero.

### 4. Review of Comments

The comments within the `MultiHeadSelfAttention` class accurately describe the masking mechanism, explaining that `mask == 0` indicates elements to ignore and that `masked_fill` is used to set these scores to `float('-inf')` before `softmax` to effectively nullify their contribution.

Therefore, the existing `MultiHeadSelfAttention` module is already capable of performing masked attention when a suitable mask (like a causal/look-ahead mask) is provided to its `forward` method.
```

## Masked Attention in Decoders

Masked attention is a crucial mechanism primarily used in the decoder component of sequence-to-sequence models, such as the Transformer. Its main purpose is to enforce **causality** and prevent the decoder from 'cheating' by looking at future tokens in the target sequence during training.

### 1. Purpose of Masked Attention

In a decoder, especially when generating a sequence token by token (e.g., in language generation or machine translation), the prediction of the current token should only depend on the previously generated tokens. If the decoder were allowed to attend to all tokens in the target sequence, including those that come after the current position, it would have access to the 'answer' and the learning process would be trivialized. This is particularly important during the training phase where the full target sequence is available.

**Key Role:** To ensure that for a given output position `t`, the attention mechanism can only consider information from positions `1` to `t` (inclusive) and not from positions `t+1` onwards.

### 2. Mathematical Application of Masking

The masking operation is applied directly to the attention scores (the output of $\mathbf{Q} \mathbf{K}^T$) *before* the softmax function. This is critical because modifying the scores before softmax ensures that these 'future' positions receive a probability of virtually zero after softmax, effectively preventing them from contributing to the weighted sum of values.

The scaled dot-product attention formula, with masking, is:

$$ \text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\left(\frac{\mathbf{Q} \mathbf{K}^T}{\sqrt{d_k}} + \mathbf{M}\right)\mathbf{V} $$

Where:
-   $\mathbf{Q} \mathbf{K}^T / \sqrt{d_k}$ are the scaled attention scores.
-   $\mathbf{M}$ is the **mask matrix**. For positions $(i, j)$ where $j > i$ (i.e., token $j$ is a future token relative to token $i$), the corresponding element $M_{ij}$ is set to $-\infty$. For all other positions ($j \le i$), $M_{ij}$ is typically set to 0. (In practice, a boolean mask might be used where `True` indicates values to keep and `False` indicates values to mask out, which then get filled with $-\infty$).

When $-\infty$ is added to the attention scores for future positions, applying the `softmax` function results in $e^{-\infty}$ which is 0. This makes the attention weights for future tokens zero, meaning they have no influence on the output for the current token.

### 3. Integration into `MultiHeadSelfAttention` Module

The `MultiHeadSelfAttention` module implemented in the previous step already includes the necessary functionality to apply masking. This is handled by the `mask` parameter in the `forward` method.

Specifically, the following lines of code within the `forward` method are responsible for applying the mask:

```python
        # ... (previous code)

        # 4. Apply Mask (if provided)
        # Typically, mask values of 0 are used to indicate elements to ignore (e.g., padding).
        # Masking is done by filling the positions to be ignored with a very small negative number
        # so that their softmax probabilities become close to zero.
        if mask is not None:
            # The mask should be broadcastable. Common shapes: (batch_size, 1, seq_len, seq_len)
            # or (1, 1, seq_len, seq_len) for decoder masking.
            attention_scores = attention_scores.masked_fill(mask == 0, float('-inf'))

        # 5. Apply Softmax to get Attention Weights
        attention_weights = torch.softmax(attention_scores, dim=-1)

        # ... (subsequent code)
```

In this implementation:
-   The `mask` parameter is a `torch.Tensor`.
-   The condition `mask == 0` is used to identify the positions that should be masked (e.g., `False` in a boolean mask, or `0` in a binary mask if we follow the convention that `0` means 'ignore').
-   `attention_scores.masked_fill(mask == 0, float('-inf'))` performs the mathematical operation described: it replaces the scaled attention scores at masked positions with a very small negative number (`-inf`), ensuring their softmax probability becomes zero.

### 4. Review of Comments

The comments within the `MultiHeadSelfAttention` class accurately describe the masking mechanism, explaining that `mask == 0` indicates elements to ignore and that `masked_fill` is used to set these scores to `float('-inf')` before `softmax` to effectively nullify their contribution.

Therefore, the existing `MultiHeadSelfAttention` module is already capable of performing masked attention when a suitable mask (like a causal/look-ahead mask) is provided to its `forward` method.


## Masked Attention in Decoders

Masked attention is a crucial mechanism primarily used in the decoder component of sequence-to-sequence models, such as the Transformer. Its main purpose is to enforce **causality** and prevent the decoder from 'cheating' by looking at future tokens in the target sequence during training.

### 1. Purpose of Masked Attention

In a decoder, especially when generating a sequence token by token (e.g., in language generation or machine translation), the prediction of the current token should only depend on the previously generated tokens. If the decoder were allowed to attend to all tokens in the target sequence, including those that come after the current position, it would have access to the 'answer' and the learning process would be trivialized. This is particularly important during the training phase where the full target sequence is available.

**Key Role:** To ensure that for a given output position `t`, the attention mechanism can only consider information from positions `1` to `t` (inclusive) and not from positions `t+1` onwards.

### 2. Mathematical Application of Masking

The masking operation is applied directly to the attention scores (the output of $\mathbf{Q} \mathbf{K}^T$) *before* the softmax function. This is critical because modifying the scores before softmax ensures that these 'future' positions receive a probability of virtually zero after softmax, effectively preventing them from contributing to the weighted sum of values.

The scaled dot-product attention formula, with masking, is:

$$ \text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\left(\frac{\mathbf{Q} \mathbf{K}^T}{\sqrt{d_k}} + \mathbf{M}\right)\mathbf{V} $$

Where:
-   $\mathbf{Q} \mathbf{K}^T / \sqrt{d_k}$ are the scaled attention scores.
-   $\mathbf{M}$ is the **mask matrix**. For positions $(i, j)$ where $j > i$ (i.e., token $j$ is a future token relative to token $i$), the corresponding element $M_{ij}$ is set to $-\infty$. For all other positions ($j \le i$), $M_{ij}$ is typically set to 0. (In practice, a boolean mask might be used where `True` indicates values to keep and `False` indicates values to mask out, which then get filled with $-\infty$).

When $-\infty$ is added to the attention scores for future positions, applying the `softmax` function results in $e^{-\infty}$ which is 0. This makes the attention weights for future tokens zero, meaning they have no influence on the output for the current token.

### 3. Integration into `MultiHeadSelfAttention` Module

The `MultiHeadSelfAttention` module implemented in the previous step already includes the necessary functionality to apply masking. This is handled by the `mask` parameter in the `forward` method.

Specifically, the following lines of code within the `forward` method are responsible for applying the mask:

```python
        # ... (previous code)

        # 4. Apply Mask (if provided)
        # Typically, mask values of 0 are used to indicate elements to ignore (e.g., padding).
        # Masking is done by filling the positions to be ignored with a very small negative number
        # so that their softmax probabilities become close to zero.
        if mask is not None:
            # The mask should be broadcastable. Common shapes: (batch_size, 1, seq_len, seq_len)
            # or (1, 1, seq_len, seq_len) for decoder masking.
            attention_scores = attention_scores.masked_fill(mask == 0, float('-inf'))

        # 5. Apply Softmax to get Attention Weights
        attention_weights = torch.softmax(attention_scores, dim=-1)

        # ... (subsequent code)
```

In this implementation:
-   The `mask` parameter is a `torch.Tensor`.
-   The condition `mask == 0` is used to identify the positions that should be masked (e.g., `False` in a boolean mask, or `0` in a binary mask if we follow the convention that `0` means 'ignore').
-   `attention_scores.masked_fill(mask == 0, float('-inf'))` performs the mathematical operation described: it replaces the scaled attention scores at masked positions with a very small negative number (`-inf`), ensuring their softmax probability becomes zero.

### 4. Review of Comments

The comments within the `MultiHeadSelfAttention` class accurately describe the masking mechanism, explaining that `mask == 0` indicates elements to ignore and that `masked_fill` is used to set these scores to `float('-inf')` before `softmax` to effectively nullify their contribution.

Therefore, the existing `MultiHeadSelfAttention` module is already capable of performing masked attention when a suitable mask (like a causal/look-ahead mask) is provided to its `forward` method.


## Layer Normalization

### Subtask:
Explain the mathematical formulation of layer normalization. Implement a PyTorch module for layer normalization, including detailed comments.


```markdown
## Mathematical Formulation of Layer Normalization

Layer Normalization (LN) is a technique used to normalize the inputs of a layer, especially in recurrent neural networks and Transformers, to stabilize and accelerate training. Unlike Batch Normalization, which normalizes across the batch dimension, Layer Normalization normalizes across the feature dimension for each individual sample in the batch. This makes it particularly suitable for variable-length sequences and smaller batch sizes, as it does not depend on batch statistics.

For an input feature vector $\mathbf{x} = (x_1, \dots, x_D)$ within a given sample (where $D$ is the number of features, e.g., `d_model` in Transformer), Layer Normalization computes the mean and variance of the features for that *single* sample.

Let the input to the layer be $\mathbf{h}$, a tensor of shape $(N, L, D)$, where $N$ is the batch size, $L$ is the sequence length, and $D$ is the feature dimension (e.g., `d_model`). Layer Normalization typically normalizes along the last dimension(s) (the features) for each $(N, L)$ position independently.

For a given input vector $\mathbf{x} \in \mathbb{R}^D$ (e.g., one token's embedding vector for one sample), the normalization process is as follows:

1.  **Calculate the Mean ($\mu$)**: The mean is computed across all elements of the feature vector $\mathbf{x}$ for each training example independently.

    $$ \mu = \frac{1}{D} \sum_{i=1}^{D} x_i $$

2.  **Calculate the Variance ($\sigma^2$)**: The variance is computed across all elements of the feature vector $\mathbf{x}$ for each training example independently.

    $$ \sigma^2 = \frac{1}{D} \sum_{i=1}^{D} (x_i - \mu)^2 $$

3.  **Normalize the Input**: Each element $x_i$ of the input vector is then normalized using the calculated mean and variance. A small epsilon ($\epsilon$) is added to the variance for numerical stability, preventing division by zero if the variance is very small.

    $$ \hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}} $$

4.  **Apply Learnable Scale ($\gamma$) and Shift ($\beta$) Parameters**: After normalization, a learnable scaling factor $\gamma$ and a learnable shifting factor $\beta$ are applied element-wise. These parameters allow the network to learn optimal scaling and shifting for the normalized values, potentially restoring the representational power lost during normalization.

    $$ y_i = \gamma \hat{x}_i + \beta $$

    Where:
    *   $\gamma \in \mathbb{R}^D$ is a learnable scale parameter (often initialized to ones).
    *   $\beta \in \mathbb{R}^D$ is a learnable shift parameter (often initialized to zeros).
    *   The operations are element-wise (Hadamard product for scaling).

In the context of PyTorch's `nn.LayerNorm`, `normalized_shape` specifies the dimensions over which the mean and variance are calculated (typically the last `d_model` dimension for Transformer blocks), and `elementwise_affine=True` enables the learnable $\gamma$ and $\beta$ parameters.

Layer Normalization helps in addressing the vanishing/exploding gradient problem, allows for higher learning rates, and makes training more stable and less sensitive to initialization, especially in deep networks.
```


**Reasoning**:
The mathematical formulation of Layer Normalization has been explained. Now, I will proceed with the PyTorch implementation of the `LayerNorm` module, including detailed comments and an example usage, as requested in the subtask.



In [ ]:
import torch
import torch.nn as nn

class LayerNorm(nn.Module):
    """
    A PyTorch module for Layer Normalization.

    Layer Normalization normalizes the inputs across the feature dimension
    for each individual sample in the batch, making it suitable for sequence models.
    It applies learnable scale (gamma) and shift (beta) parameters after normalization.
    """
    def __init__(self, normalized_shape: int, eps: float = 1e-5):
        """
        Initializes the LayerNorm layer.

        Args:
            normalized_shape (int): The dimension over which to normalize (e.g., d_model in Transformer).
                                    This is typically the last dimension of the input tensor.
            eps (float): A small value added to the variance for numerical stability.
        """
        super().__init__()
        self.normalized_shape = (normalized_shape,) # Needs to be a tuple for consistency with nn.LayerNorm
        self.eps = eps

        # Learnable parameters: gamma (scale) and beta (shift)
        # Initialized to ones for gamma and zeros for beta, to start with identity transformation.
        self.gamma = nn.Parameter(torch.ones(normalized_shape))
        self.beta = nn.Parameter(torch.zeros(normalized_shape))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Performs the forward pass of the LayerNorm layer.

        Args:
            x (torch.Tensor): The input tensor of any shape, where the last dimension
                              is `normalized_shape`. E.g., (batch_size, sequence_length, d_model).

        Returns:
            torch.Tensor: The normalized output tensor with the same shape as the input.
        """
        # Calculate mean and variance across the last dimension (feature dimension)
        # for each example independently.
        # keepdim=True ensures that the output tensors for mean/variance have the same
        # number of dimensions as x, facilitating broadcasting.
        mean = x.mean(dim=-1, keepdim=True)
        variance = x.var(dim=-1, keepdim=True, unbiased=False) # unbiased=False for population variance

        # Normalize the input: (x - mean) / sqrt(variance + epsilon)
        normalized_x = (x - mean) / torch.sqrt(variance + self.eps)

        # Apply learnable gamma (scale) and beta (shift) parameters
        # These are broadcasted across the dimensions preceding the last one.
        output = self.gamma * normalized_x + self.beta

        return output

# Example Usage and Verification:
# Define parameters
example_d_model = 256
example_batch_size = 2
example_sequence_length = 10

# Instantiate LayerNorm layer
layer_norm = LayerNorm(normalized_shape=example_d_model)
print(f"LayerNorm layer created with normalized_shape={example_d_model}")
print(f"Gamma parameter shape: {layer_norm.gamma.shape}")
print(f"Beta parameter shape: {layer_norm.beta.shape}")

# Create a sample input tensor (e.g., from MultiHeadSelfAttention output)
example_input_tensor = torch.randn(example_batch_size, example_sequence_length, example_d_model)
print(f"\nExample input tensor shape: {example_input_tensor.shape}")

# Perform forward pass
output_tensor = layer_norm(example_input_tensor)
print(f"Output tensor shape: {output_tensor.shape}")

# Verify output dimensions match expected
assert output_tensor.shape == (example_batch_size, example_sequence_length, example_d_model)
print("Verification successful: Output shape is correct.")

# Further verification: check if mean is approx 0 and variance is approx 1 over the normalized dimension
# for a specific sample and token after normalization (before gamma/beta)
# This is done by taking the mean and variance of the *unscaled* normalized_x

# To do this, we need to manually compute the unscaled normalized_x within a test block.
# For a single sample and token:
sample_input = example_input_tensor[0, 0, :]
mean_manual = sample_input.mean(dim=-1, keepdim=True)
var_manual = sample_input.var(dim=-1, keepdim=True, unbiased=False)
normalized_manual = (sample_input - mean_manual) / torch.sqrt(var_manual + layer_norm.eps)

# Check mean and variance of the manually normalized tensor
print(f"\nMean of manually normalized vector (first token, first sample): {normalized_manual.mean().item():.4f}")
print(f"Variance of manually normalized vector (first token, first sample): {normalized_manual.var(unbiased=False).item():.4f}")
assert torch.isclose(normalized_manual.mean(), torch.tensor(0.0), atol=1e-4)
assert torch.isclose(normalized_manual.var(unbiased=False), torch.tensor(1.0), atol=1e-4)
print("Verification successful: Normalized mean is ~0 and variance is ~1.")


LayerNorm layer created with normalized_shape=256
Gamma parameter shape: torch.Size([256])
Beta parameter shape: torch.Size([256])

Example input tensor shape: torch.Size([2, 10, 256])
Output tensor shape: torch.Size([2, 10, 256])
Verification successful: Output shape is correct.

Mean of manually normalized vector (first token, first sample): 0.0000
Variance of manually normalized vector (first token, first sample): 1.0000
Verification successful: Normalized mean is ~0 and variance is ~1.


## Feed-Forward Network

### Subtask:
Explain the mathematical formulation of a simple feed-forward network (two linear layers with an activation). Implement a PyTorch module for this FFN, including detailed comments.


```markdown
## Mathematical Formulation of a Feed-Forward Network (FFN)

In the Transformer architecture, a Feed-Forward Network (FFN), also known as a Position-wise Feed-Forward Network, is applied independently and identically to each position in the sequence. It consists of two linear transformations with a non-linear activation function in between.

For an input sequence $\mathbf{X} \in \mathbb{R}^{L \times d_{model}}$, where $L$ is the sequence length and $d_{model}$ is the dimensionality of each token's representation, the FFN processes each position $x \in \mathbb{R}^{d_{model}}$ independently. This means that for each position, the same FFN parameters are used.

Let $x \in \mathbb{R}^{d_{model}}$ be the input vector for a single position (e.g., from the output of a multi-head attention layer). The FFN computation for this position is defined as follows:

1.  **First Linear Transformation**: The input vector $x$ is projected into a higher-dimensional space. This transformation uses a weight matrix $\mathbf{W}_1 \in \mathbb{R}^{d_{model} \times d_{ff}}$ and a bias vector $\mathbf{b}_1 \in \mathbb{R}^{d_{ff}}$.

    $$ \mathbf{h} = \text{Linear}_1(x) = x \mathbf{W}_1 + \mathbf{b}_1 $$

    Where $d_{ff}$ is the hidden layer dimensionality of the feed-forward network, which is typically larger than $d_{model}$ (e.g., $d_{ff} = 4 \cdot d_{model}$). The output $\mathbf{h} \in \mathbb{R}^{d_{ff}}$ has the expanded dimension.

2.  **Activation Function**: A non-linear activation function is applied element-wise to the output of the first linear transformation. The original Transformer paper uses the Rectified Linear Unit (ReLU) function, but other activations like GELU are also common.

    $$ \mathbf{h}' = \text{ReLU}(\mathbf{h}) = \max(0, \mathbf{h}) $$

    Where $\text{ReLU}$ is applied element-wise.

3.  **Second Linear Transformation**: The activated vector $\mathbf{h}'$ is then projected back to the original $d_{model}$ dimensionality. This transformation uses a weight matrix $\mathbf{W}_2 \in \mathbb{R}^{d_{ff} \times d_{model}}$ and a bias vector $\mathbf{b}_2 \in \mathbb{R}^{d_{model}}$.

    $$ \text{FFN}(x) = \text{Linear}_2(\mathbf{h}') = \mathbf{h}' \mathbf{W}_2 + \mathbf{b}_2 $$

    The final output $\text{FFN}(x) \in \mathbb{R}^{d_{model}}$ has the same dimensionality as the input $x$.

Combining these steps, the complete mathematical formulation for a single position is:

$$ \text{FFN}(x) = \max(0, x \mathbf{W}_1 + \mathbf{b}_1) \mathbf{W}_2 + \mathbf{b}_2 $$

When applied to the entire sequence $\mathbf{X}$, the operations are performed in a vectorized manner across the sequence length dimension:

$$ \text{FFN}(\mathbf{X}) = \text{ReLU}(\mathbf{X} \mathbf{W}_1 + \mathbf{b}_1) \mathbf{W}_2 + \mathbf{b}_2 $$

Where the bias vectors are broadcast across the sequence length dimension.

This FFN allows the model to capture complex, non-linear relationships within each token's representation, enriching its feature extraction capabilities.
```

**Reasoning**:
The mathematical formulation of the Feed-Forward Network has been explained. Now, I will implement the PyTorch module for the FFN, including detailed comments and an example usage, as requested in the subtask instructions.



In [ ]:
import torch
import torch.nn as nn

class FeedForwardNetwork(nn.Module):
    """
    A PyTorch module for a simple Feed-Forward Network (FFN).

    This FFN consists of two linear layers with a ReLU activation function in between.
    It is applied independently to each position in the sequence.
    """
    def __init__(self, d_model: int, d_ff: int, dropout_rate: float = 0.1):
        """
        Initializes the FeedForwardNetwork layer.

        Args:
            d_model (int): The dimensionality of the input and output embeddings.
                           This is the model's internal representation dimension.
            d_ff (int): The hidden layer dimensionality of the feed-forward network.
                        Typically, d_ff is 4 times d_model (e.g., 2048 for d_model=512).
            dropout_rate (float): The dropout probability to apply after the activation.
        """
        super().__init__()
        # First linear layer: projects input from d_model to d_ff (expansion)
        # W_1 in the mathematical formulation.
        self.linear_1 = nn.Linear(d_model, d_ff)

        # Activation function: ReLU as per the original Transformer paper
        self.relu = nn.ReLU()

        # Second linear layer: projects from d_ff back to d_model (contraction)
        # W_2 in the mathematical formulation.
        self.linear_2 = nn.Linear(d_ff, d_model)

        # Dropout layer to prevent overfitting
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Performs the forward pass of the FeedForwardNetwork layer.

        Args:
            x (torch.Tensor): The input tensor of shape (batch_size, sequence_length, d_model).
                              This typically comes from the output of an attention sub-layer.

        Returns:
            torch.Tensor: The output tensor of the FFN, with shape (batch_size, sequence_length, d_model).
        """
        # Apply the first linear transformation
        # Input: (batch_size, seq_len, d_model) -> Output: (batch_size, seq_len, d_ff)
        x = self.linear_1(x)

        # Apply the ReLU activation function element-wise
        # Output: (batch_size, seq_len, d_ff)
        x = self.relu(x)

        # Apply dropout for regularization
        x = self.dropout(x)

        # Apply the second linear transformation
        # Input: (batch_size, seq_len, d_ff) -> Output: (batch_size, seq_len, d_model)
        x = self.linear_2(x)

        return x

# Example Usage and Verification:
# Define parameters
example_d_model = 256  # Input and output dimension
example_d_ff = 1024    # Inner dimension, typically 4 * d_model
example_batch_size = 2
example_sequence_length = 10

# Instantiate the FeedForwardNetwork layer
ffn_layer = FeedForwardNetwork(d_model=example_d_model, d_ff=example_d_ff)
print(f"FeedForwardNetwork created with d_model={example_d_model}, d_ff={example_d_ff}")

# Create a sample input tensor
example_input_tensor = torch.randn(example_batch_size, example_sequence_length, example_d_model)
print(f"\nExample input tensor shape: {example_input_tensor.shape}")

# Pass the input through the FFN module
output_tensor = ffn_layer(example_input_tensor)

print(f"Output tensor shape: {output_tensor.shape}")

# Verify output dimensions match expected (should be same as input dimensions)
expected_output_shape = (example_batch_size, example_sequence_length, example_d_model)
assert output_tensor.shape == expected_output_shape
print("Verification successful: Output shape is correct and matches input shape.")


FeedForwardNetwork created with d_model=256, d_ff=1024

Example input tensor shape: torch.Size([2, 10, 256])
Output tensor shape: torch.Size([2, 10, 256])
Verification successful: Output shape is correct and matches input shape.


## Residual Connections

### Subtask:
Explain the concept and importance of residual connections in Transformers. Integrate residual connections within the context of combining attention and FFN layers, illustrating their placement and function.


```markdown
## Residual Connections (Skip Connections) in Transformers

Residual connections, often referred to as skip connections, are a fundamental concept in deep learning, especially critical for training very deep neural networks like the Transformer. They provide a direct pathway for information to flow through the network, bypassing one or more layers.

### 1. Mathematical Formulation and Purpose

Mathematically, a residual connection works by adding the input of a sub-layer to its output. If we consider a sub-layer function $F(x)$ (e.g., a Multi-Head Self-Attention or a Feed-Forward Network), the output with a residual connection is:

$$ \text{Output} = F(x) + x $$

Where $x$ is the input to the sub-layer.

**Purpose and Importance:**

*   **Mitigating Vanishing/Exploding Gradients**: In very deep networks, gradients can become extremely small (vanish) or very large (explode) as they are backpropagated through many layers. Residual connections create direct paths for gradients to flow backward, effectively creating an ensemble of paths of varying lengths, which helps to stabilize training and ensures that earlier layers receive meaningful gradient updates.
*   **Enabling Deeper Models**: By allowing layers to learn residual functions (i.e., the difference $F(x)$ rather than the complete transformation $F(x)+x$), it becomes easier for deeper networks to train. If a layer finds that its optimal function is simply the identity function (passing the input through unchanged), it can easily learn $F(x) = 0$ due to the additive nature of the connection. Without residual connections, learning an identity mapping through multiple non-linear layers is much harder.
*   **Improved Information Flow**: They ensure that the original input information is always preserved and directly contributes to the output, even as it passes through multiple transformations. This helps maintain signal integrity and allows layers to refine representations rather than completely re-learn them.

### 2. Application in Transformer Blocks: Add & Norm

In the Transformer architecture, residual connections are applied around every sub-layer (the multi-head attention and the feed-forward network), followed by layer normalization. This combination is often called **"Add & Norm"**.

For each sub-layer, the process is:

1.  **Sub-layer computation**: $Y = \text{Sublayer}(X)$
2.  **Residual Connection**: The input $X$ is added to the output $Y$. This is the "Add" part.
    $$ Z = X + Y $$
3.  **Layer Normalization**: The result $Z$ is then passed through a Layer Normalization layer. This is the "Norm" part.
    $$ \text{Output} = \text{LayerNorm}(Z) = \text{LayerNorm}(X + \text{Sublayer}(X)) $$

**Placement and Function:**

*   **Before Layer Normalization**: The residual connection is applied *before* layer normalization. This means that the normalization step operates on the sum of the input and the sub-layer's output. This sequence (Add then Norm) is crucial and is often preferred over "Norm then Add" as it typically leads to more stable training and better performance.
*   **After each sub-layer**: In a standard Transformer decoder block, there are typically two main sub-layers where residual connections are applied:
    1.  **Masked Multi-Head Self-Attention**: The output of this attention mechanism is added to its input, and then layer-normalized.
    2.  **Feed-Forward Network**: The output of the FFN is added to its input (which is the output of the first Add & Norm block), and then layer-normalized again.

This structure ensures that each sub-layer processes information, and its refined output is always combined with the original (or previously transformed) input in a stable and gradient-friendly manner before further processing. The Layer Normalization step then helps to re-center and re-scale the combined features, preventing activations from growing too large or too small, and thereby further aiding training stability.
```


**Reasoning**:
The mathematical explanation for residual connections and their 'Add & Norm' application in Transformers has been provided. Now, I will implement a PyTorch module that encapsulates this common pattern (a sublayer followed by a residual connection and layer normalization) to be reused in the Transformer block.



In [ ]:
import torch
import torch.nn as nn

# Assuming LayerNorm class is already defined from a previous step
# If not, it should be included or imported.
# For demonstration, we'll use the previously defined LayerNorm:

class LayerNorm(nn.Module):
    """
    A PyTorch module for Layer Normalization.

    Layer Normalization normalizes the inputs across the feature dimension
    for each individual sample in the batch, making it suitable for sequence models.
    It applies learnable scale (gamma) and shift (beta) parameters after normalization.
    """
    def __init__(self, normalized_shape: int, eps: float = 1e-5):
        super().__init__()
        self.normalized_shape = (normalized_shape,)
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(normalized_shape))
        self.beta = nn.Parameter(torch.zeros(normalized_shape))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mean = x.mean(dim=-1, keepdim=True)
        variance = x.var(dim=-1, keepdim=True, unbiased=False)
        normalized_x = (x - mean) / torch.sqrt(variance + self.eps)
        output = self.gamma * normalized_x + self.beta
        return output


class ResidualConnection(nn.Module):
    """
    A PyTorch module that applies a residual connection followed by layer normalization.
    This implements the "Add & Norm" step commonly found in Transformer blocks.
    """
    def __init__(self, d_model: int, dropout_rate: float = 0.1):
        """
        Initializes the ResidualConnection module.

        Args:
            d_model (int): The dimensionality of the input and output (embedding dimension).
            dropout_rate (float): The dropout probability applied to the sublayer's output
                                  before adding it to the input.
        """
        super().__init__()
        self.dropout = nn.Dropout(dropout_rate)
        self.norm = LayerNorm(normalized_shape=d_model) # Use our custom LayerNorm

    def forward(self, x: torch.Tensor, sublayer_output: torch.Tensor) -> torch.Tensor:
        """
        Performs the forward pass of the ResidualConnection.

        Args:
            x (torch.Tensor): The input to the sublayer (and to the residual connection).
                              Shape: (batch_size, sequence_length, d_model).
            sublayer_output (torch.Tensor): The output from the sublayer (e.g., attention or FFN).
                                            Shape: (batch_size, sequence_length, d_model).

        Returns:
            torch.Tensor: The output after adding the residual connection and applying layer normalization.
                          Shape: (batch_size, sequence_length, d_model).
        """
        # 1. Apply dropout to the sublayer's output
        # This is typically applied to the output of the attention or FFN sublayer
        # before it's added to the input, as per the original Transformer paper.
        sublayer_output = self.dropout(sublayer_output)

        # 2. Add the residual connection
        # This is the "Add" part: input + sublayer_output
        # Ensures information flows directly from the input to the output of the block.
        added_output = x + sublayer_output

        # 3. Apply Layer Normalization
        # This is the "Norm" part: LayerNorm(input + sublayer_output)
        # Normalizes the combined output across the feature dimension.
        output = self.norm(added_output)

        return output

# Example Usage and Verification:
# Define parameters
example_d_model = 256
example_batch_size = 2
example_sequence_length = 10

# Instantiate ResidualConnection layer
residual_conn = ResidualConnection(d_model=example_d_model)
print(f"ResidualConnection layer created with d_model={example_d_model}")

# Create a sample input tensor 'x' (e.g., embeddings before a sublayer)
example_input_x = torch.randn(example_batch_size, example_sequence_length, example_d_model)
print(f"\nExample input 'x' shape: {example_input_x.shape}")

# Create a sample sublayer output tensor (e.g., output of attention or FFN)
example_sublayer_out = torch.randn(example_batch_size, example_sequence_length, example_d_model)
print(f"Example sublayer output shape: {example_sublayer_out.shape}")

# Perform forward pass
output_with_residual = residual_conn(example_input_x, example_sublayer_out)
print(f"Output with residual connection shape: {output_with_residual.shape}")

# Verify output dimensions match expected
expected_output_shape = (example_batch_size, example_sequence_length, example_d_model)
assert output_with_residual.shape == expected_output_shape
print("Verification successful: Output shape is correct.")

# Quick check to ensure the 'add' part is working (before normalization and dropout)
# Let's manually calculate the 'added_output' and compare
dropped_sublayer_out = residual_conn.dropout(example_sublayer_out)
manually_added_output = example_input_x + dropped_sublayer_out
# We cannot directly compare output_with_residual to manually_added_output
# because output_with_residual has been layer-normalized. But we can ensure
# the normalization happens on the sum.

# Let's check if mean and variance are normalized for the first element after 'Add & Norm'
# (this is already tested by LayerNorm, but confirms the sequence)
mean_after_norm = output_with_residual[0, 0, :].mean().item()
var_after_norm = output_with_residual[0, 0, :].var(unbiased=False).item()
print(f"\nMean of output features (first token, first sample) after Add & Norm: {mean_after_norm:.4f}")
print(f"Variance of output features (first token, first sample) after Add & Norm: {var_after_norm:.4f}")
assert torch.isclose(torch.tensor(mean_after_norm), torch.tensor(0.0), atol=1e-4)
assert torch.isclose(torch.tensor(var_after_norm), torch.tensor(1.0), atol=1e-4)
print("Verification successful: Output of ResidualConnection has normalized features.")

ResidualConnection layer created with d_model=256

Example input 'x' shape: torch.Size([2, 10, 256])
Example sublayer output shape: torch.Size([2, 10, 256])
Output with residual connection shape: torch.Size([2, 10, 256])
Verification successful: Output shape is correct.

Mean of output features (first token, first sample) after Add & Norm: 0.0000
Variance of output features (first token, first sample) after Add & Norm: 1.0000
Verification successful: Output of ResidualConnection has normalized features.


## Decoder Block Implementation

### Subtask:
Combine the multi-head masked self-attention, layer normalization, feed-forward network, and residual connections to form a single decoder block in PyTorch, including detailed comments.


**Reasoning**:
I will implement the `DecoderBlock` module, combining the previously defined `MultiHeadSelfAttention`, `FeedForwardNetwork`, and `ResidualConnection` modules, along with a causal mask, to create a complete decoder block in PyTorch, as requested in the subtask.



In [ ]:
import torch
import torch.nn as nn
import math

# Re-importing necessary components for a self-contained example
# (In a real project, these would be imported from their respective files)

class LayerNorm(nn.Module):
    """
    A PyTorch module for Layer Normalization.

    Layer Normalization normalizes the inputs across the feature dimension
    for each individual sample in the batch, making it suitable for sequence models.
    It applies learnable scale (gamma) and shift (beta) parameters after normalization.
    """
    def __init__(self, normalized_shape: int, eps: float = 1e-5):
        super().__init__()
        self.normalized_shape = (normalized_shape,)
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(normalized_shape))
        self.beta = nn.Parameter(torch.zeros(normalized_shape))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mean = x.mean(dim=-1, keepdim=True)
        variance = x.var(dim=-1, keepdim=True, unbiased=False)
        normalized_x = (x - mean) / torch.sqrt(variance + self.eps)
        output = self.gamma * normalized_x + self.beta
        return output


class MultiHeadSelfAttention(nn.Module):
    """
    A PyTorch module for Multi-Head Self-Attention (MHSA).

    MHSA allows the model to jointly attend to information from different
    representation subspaces at different positions. It uses multiple
    "heads" to perform attention in parallel, and then concatenates their
    results and projects them back.
    """
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError(f"d_model ({d_model}) must be divisible by num_heads ({num_heads})")

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.query_proj = nn.Linear(d_model, d_model)
        self.key_proj = nn.Linear(d_model, d_model)
        self.value_proj = nn.Linear(d_model, d_model)
        self.output_proj = nn.Linear(d_model, d_model)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        batch_size, sequence_length, _ = x.size()

        Q = self.query_proj(x)
        K = self.key_proj(x)
        V = self.value_proj(x)

        Q = Q.view(batch_size, sequence_length, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, sequence_length, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, sequence_length, self.num_heads, self.head_dim).transpose(1, 2)

        attention_scores = torch.matmul(Q, K.transpose(-2, -1))
        attention_scores = attention_scores / math.sqrt(self.head_dim)

        if mask is not None:
            attention_scores = attention_scores.masked_fill(mask == 0, float('-inf'))

        attention_weights = torch.softmax(attention_scores, dim=-1)
        output = torch.matmul(attention_weights, V)

        output = output.transpose(1, 2).contiguous().view(batch_size, sequence_length, self.d_model)
        output = self.output_proj(output)

        return output


class FeedForwardNetwork(nn.Module):
    """
    A PyTorch module for a simple Feed-Forward Network (FFN).

    This FFN consists of two linear layers with a ReLU activation function in between.
    It is applied independently to each position in the sequence.
    """
    def __init__(self, d_model: int, d_ff: int, dropout_rate: float = 0.1):
        super().__init__()
        self.linear_1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.linear_2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear_1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.linear_2(x)
        return x


class ResidualConnection(nn.Module):
    """
    A PyTorch module that applies a residual connection followed by layer normalization.
    This implements the "Add & Norm" step commonly found in Transformer blocks.
    """
    def __init__(self, d_model: int, dropout_rate: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout_rate)
        self.norm = LayerNorm(normalized_shape=d_model)

    def forward(self, x: torch.Tensor, sublayer_output: torch.Tensor) -> torch.Tensor:
        sublayer_output = self.dropout(sublayer_output)
        added_output = x + sublayer_output
        output = self.norm(added_output)
        return output


class DecoderBlock(nn.Module):
    """
    A single Decoder Block for the Transformer architecture.

    This block consists of a masked multi-head self-attention sub-layer,
    followed by a position-wise feed-forward network. Each sub-layer
    includes a residual connection and layer normalization.
    """
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout_rate: float = 0.1):
        """
        Initializes the DecoderBlock.

        Args:
            d_model (int): The dimensionality of the input and output embeddings (model dimension).
            num_heads (int): The number of attention heads in the MultiHeadSelfAttention.
            d_ff (int): The hidden layer dimensionality of the FeedForwardNetwork.
            dropout_rate (float): The dropout probability used in ResidualConnection and FFN.
        """
        super().__init__()

        # Masked Multi-Head Self-Attention sub-layer
        # This attention mechanism allows a token to attend only to previous tokens in the sequence.
        self.masked_self_attention = MultiHeadSelfAttention(d_model, num_heads)

        # First Residual Connection + Layer Normalization
        # Applied after the masked self-attention. Takes the input and adds the attention output.
        self.residual_conn1 = ResidualConnection(d_model, dropout_rate)

        # Position-wise Feed-Forward Network
        # Applied to each position independently after the first Add&Norm.
        self.feed_forward = FeedForwardNetwork(d_model, d_ff, dropout_rate)

        # Second Residual Connection + Layer Normalization
        # Applied after the feed-forward network. Takes the output of the first ResidualConnection
        # and adds the FFN output.
        self.residual_conn2 = ResidualConnection(d_model, dropout_rate)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        """
        Performs the forward pass of the DecoderBlock.

        Args:
            x (torch.Tensor): The input tensor to the decoder block.
                              Shape: (batch_size, sequence_length, d_model).
            mask (torch.Tensor, optional): The causal (look-ahead) mask for masked self-attention.
                                           Shape: (1, 1, sequence_length, sequence_length)
                                           or (batch_size, 1, sequence_length, sequence_length).

        Returns:
            torch.Tensor: The output tensor of the decoder block.
                          Shape: (batch_size, sequence_length, d_model).
        """
        # Store the input 'x' for the first residual connection
        # input_to_attention will be used as the 'x' argument for residual_conn1
        input_to_attention = x

        # 1. Masked Multi-Head Self-Attention
        # Apply masked self-attention to the input. The mask prevents attending to future tokens.
        # Input: (batch_size, seq_len, d_model) -> Output: (batch_size, seq_len, d_model)
        attention_output = self.masked_self_attention(x, mask)

        # 2. First Residual Connection and Layer Normalization
        # Add 'input_to_attention' to 'attention_output' and then normalize.
        # Output: (batch_size, seq_len, d_model)
        x = self.residual_conn1(input_to_attention, attention_output)

        # Store the output of the first Add&Norm for the second residual connection
        # input_to_ffn will be used as the 'x' argument for residual_conn2
        input_to_ffn = x

        # 3. Position-wise Feed-Forward Network
        # Pass the output of the first Add&Norm through the FFN.
        # Input: (batch_size, seq_len, d_model) -> Output: (batch_size, seq_len, d_model)
        ffn_output = self.feed_forward(x)

        # 4. Second Residual Connection and Layer Normalization
        # Add 'input_to_ffn' to 'ffn_output' and then normalize.
        # Output: (batch_size, seq_len, d_model)
        output = self.residual_conn2(input_to_ffn, ffn_output)

        return output

# Example Usage and Verification:
# Define parameters for the decoder block
example_d_model = 256
example_num_heads = 8
example_d_ff = 1024 # Typically 4 * d_model
example_dropout_rate = 0.1
example_batch_size = 2
example_sequence_length = 10

# Instantiate the DecoderBlock
decoder_block = DecoderBlock(
    d_model=example_d_model,
    num_heads=example_num_heads,
    d_ff=example_d_ff,
    dropout_rate=example_dropout_rate
)
print(f"DecoderBlock created with d_model={example_d_model}, num_heads={example_num_heads}, d_ff={example_d_ff}")

# Create a sample input tensor
# This would typically be token embeddings + positional encodings
example_input_tensor = torch.randn(example_batch_size, example_sequence_length, example_d_model)
print(f"\nExample input tensor shape: {example_input_tensor.shape}")

# Create a causal mask for masked self-attention
# A causal mask is a lower triangular matrix where elements (i, j) with j > i are masked (set to 0).
causal_mask = torch.tril(torch.ones(example_sequence_length, example_sequence_length)).bool()
causal_mask = causal_mask.unsqueeze(0).unsqueeze(0) # Add batch and head dimensions for broadcasting
print(f"Example causal mask shape: {causal_mask.shape}")

# Perform forward pass through the DecoderBlock
output_decoder_block = decoder_block(example_input_tensor, mask=causal_mask)
print(f"Output of DecoderBlock shape: {output_decoder_block.shape}")

# Verify output dimensions match expected
expected_output_shape = (example_batch_size, example_sequence_length, example_d_model)
assert output_decoder_block.shape == expected_output_shape
print("Verification successful: Output shape of DecoderBlock is correct.")

DecoderBlock created with d_model=256, num_heads=8, d_ff=1024

Example input tensor shape: torch.Size([2, 10, 256])
Example causal mask shape: torch.Size([1, 1, 10, 10])
Output of DecoderBlock shape: torch.Size([2, 10, 256])
Verification successful: Output shape of DecoderBlock is correct.


## Final Linear Output Layer

### Subtask:
Explain the mathematical formulation of the final linear output layer that maps the decoder's output to logit scores for the vocabulary. Implement this layer in PyTorch, including detailed comments.


## Mathematical Formulation of the Final Linear Output Layer

The final linear output layer in a decoder-only Transformer is responsible for transforming the continuous, high-dimensional representations produced by the stack of decoder blocks into discrete probability distributions over the vocabulary. Specifically, it maps the `d_model`-dimensional output for each token position to a set of unnormalized logit scores, one for each possible token in the vocabulary.

Let $\mathbf{H} \in \mathbb{R}^{L \times d_{model}}$ be the output of the final decoder block, where $L$ is the sequence length and $d_{model}$ is the dimensionality of the model's hidden states. Each row $\mathbf{h}_j \in \mathbb{R}^{d_{model}}$ corresponds to the contextualized representation of the $j$-th token in the sequence.

For each token representation $\mathbf{h}_j$, the final linear layer performs a linear transformation to produce a vector of logit scores $\mathbf{l}_j \in \mathbb{R}^{|V|}$, where $|V|$ is the size of the vocabulary. This transformation is defined as:

$$ \mathbf{l}_j = \mathbf{h}_j \mathbf{W}_{out} + \mathbf{b}_{out} $$

Where:
-   $\mathbf{W}_{out} \in \mathbb{R}^{d_{model} \times |V|}$ is the learnable weight matrix that maps the `d_model` dimension to the `vocab_size` dimension.
-   $\mathbf{b}_{out} \in \mathbb{R}^{|V|}$ is the learnable bias vector.
-   $\mathbf{l}_j$ is the vector of unnormalized logit scores for the $j$-th token position. Each element $l_{j,k}$ in $\mathbf{l}_j$ represents the raw score for the $k$-th token in the vocabulary to be the next token at position $j$.

In a vectorized form, for the entire sequence output $\mathbf{H}$:

$$ \mathbf{L} = \mathbf{H} \mathbf{W}_{out} + \mathbf{B}_{out} $$

Where $\mathbf{L} \in \mathbb{R}^{L \times |V|}$ and $\mathbf{B}_{out}$ is the bias vector broadcast across the sequence length.

After obtaining these logit scores, a `softmax` function is typically applied (often as part of the loss function during training, or explicitly during inference) to convert them into a probability distribution over the vocabulary for each position:

$$ \text{Probabilities}_j = \text{softmax}(\mathbf{l}_j) $$

The token with the highest probability can then be selected as the predicted next token. The purpose of this layer is to translate the rich, abstract semantic information encoded in the `d_model`-dimensional vectors into a concrete prediction for the next word or token.

**Reasoning**:
The mathematical formulation has been explained. Now, I will implement the PyTorch module for the final linear output layer, including detailed comments, and provide example usage and verification steps as requested in the subtask instructions.



In [ ]:
import torch
import torch.nn as nn

class FinalLinearOutput(nn.Module):
    """
    A PyTorch module for the final linear output layer of a decoder-only Transformer.

    This layer maps the d_model-dimensional output of the last decoder block
    to a vocabulary_size-dimensional vector of logit scores for each token position.
    """
    def __init__(self, d_model: int, vocab_size: int):
        """
        Initializes the FinalLinearOutput layer.

        Args:
            d_model (int): The dimensionality of the input from the last decoder block.
                           This is the model's internal representation dimension.
            vocab_size (int): The total number of unique tokens in the vocabulary.
                              The output will have a logit score for each possible token.
        """
        super().__init__()
        # The linear layer performs the transformation from d_model to vocab_size.
        # This layer does not include softmax, as softmax is often combined with
        # the loss function (e.g., F.cross_entropy) for numerical stability and efficiency.
        self.linear = nn.Linear(d_model, vocab_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Performs the forward pass of the FinalLinearOutput layer.

        Args:
            x (torch.Tensor): The input tensor, typically the output of the last
                              decoder block. Shape: (batch_size, sequence_length, d_model).

        Returns:
            torch.Tensor: A tensor of unnormalized logit scores for each token in the
                          vocabulary at each position. Shape: (batch_size, sequence_length, vocab_size).
        """
        # Apply the linear transformation.
        # Input: (batch_size, sequence_length, d_model)
        # Output: (batch_size, sequence_length, vocab_size)
        logits = self.linear(x)
        return logits

# Example Usage and Verification:
# Define parameters
example_d_model = 256     # Dimensionality of the decoder's output
example_vocab_size = 10000 # Size of the vocabulary
example_batch_size = 2
example_sequence_length = 10

# Instantiate the FinalLinearOutput layer
final_output_layer = FinalLinearOutput(d_model=example_d_model, vocab_size=example_vocab_size)
print(f"FinalLinearOutput layer created with d_model={example_d_model}, vocab_size={example_vocab_size}")

# Create a sample input tensor (e.g., output from the last DecoderBlock)
example_input_tensor = torch.randn(example_batch_size, example_sequence_length, example_d_model)
print(f"\nExample input tensor shape: {example_input_tensor.shape}")

# Pass the input through the final linear output layer
output_logits = final_output_layer(example_input_tensor)

print(f"Output logits tensor shape: {output_logits.shape}")

# Verify output dimensions match expected
expected_output_shape = (example_batch_size, example_sequence_length, example_vocab_size)
assert output_logits.shape == expected_output_shape
print("Verification successful: Output shape is correct and matches (batch_size, sequence_length, vocab_size).")

# You would typically apply a softmax here to get probabilities, e.g.:
# probabilities = torch.softmax(output_logits, dim=-1)
# print(f"\nProbabilities tensor shape: {probabilities.shape}")
# print(f"Sum of probabilities for the first token in the first sequence: {probabilities[0, 0].sum().item():.4f}")

FinalLinearOutput layer created with d_model=256, vocab_size=10000

Example input tensor shape: torch.Size([2, 10, 256])
Output logits tensor shape: torch.Size([2, 10, 10000])
Verification successful: Output shape is correct and matches (batch_size, sequence_length, vocab_size).


## Full Decoder-Only Transformer

### Subtask:
Assemble all implemented components (token embedding, positional encoding, multiple decoder blocks, and the final linear layer) into a complete decoder-only Transformer model in PyTorch, including detailed comments.


**Reasoning**:
I will implement the complete `DecoderOnlyTransformer` module by combining the `TokenEmbedding`, `PositionalEncoding`, multiple `DecoderBlock`s, and `FinalLinearOutput` modules. This will involve defining the `__init__` and `forward` methods, creating a causal mask, and adding detailed comments as specified in the instructions.



In [1]:
import torch
import torch.nn as nn
import math

# Re-importing all necessary components for a self-contained example
# In a real project, these would be imported from their respective files.

class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size: int, embedding_dim: int):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        return self.embedding(token_ids)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, :x.size(1)]


class LayerNorm(nn.Module):
    def __init__(self, normalized_shape: int, eps: float = 1e-5):
        super().__init__()
        self.normalized_shape = (normalized_shape,)
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(normalized_shape))
        self.beta = nn.Parameter(torch.zeros(normalized_shape))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mean = x.mean(dim=-1, keepdim=True)
        variance = x.var(dim=-1, keepdim=True, unbiased=False)
        normalized_x = (x - mean) / torch.sqrt(variance + self.eps)
        output = self.gamma * normalized_x + self.beta
        return output


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError(f"d_model ({d_model}) must be divisible by num_heads ({num_heads})")

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.query_proj = nn.Linear(d_model, d_model)
        self.key_proj = nn.Linear(d_model, d_model)
        self.value_proj = nn.Linear(d_model, d_model)
        self.output_proj = nn.Linear(d_model, d_model)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        batch_size, sequence_length, _ = x.size()

        Q = self.query_proj(x)
        K = self.key_proj(x)
        V = self.value_proj(x)

        Q = Q.view(batch_size, sequence_length, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, sequence_length, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, sequence_length, self.num_heads, self.head_dim).transpose(1, 2)

        attention_scores = torch.matmul(Q, K.transpose(-2, -1))
        attention_scores = attention_scores / math.sqrt(self.head_dim)

        if mask is not None:
            attention_scores = attention_scores.masked_fill(mask == 0, float('-inf'))

        attention_weights = torch.softmax(attention_scores, dim=-1)
        output = torch.matmul(attention_weights, V)

        output = output.transpose(1, 2).contiguous().view(batch_size, sequence_length, self.d_model)
        output = self.output_proj(output)

        return output


class FeedForwardNetwork(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout_rate: float = 0.1):
        super().__init__()
        self.linear_1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.linear_2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear_1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.linear_2(x)
        return x


class ResidualConnection(nn.Module):
    def __init__(self, d_model: int, dropout_rate: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout_rate)
        self.norm = LayerNorm(normalized_shape=d_model)

    def forward(self, x: torch.Tensor, sublayer_output: torch.Tensor) -> torch.Tensor:
        sublayer_output = self.dropout(sublayer_output)
        added_output = x + sublayer_output
        output = self.norm(added_output)
        return output


class DecoderBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout_rate: float = 0.1):
        super().__init__()
        self.masked_self_attention = MultiHeadSelfAttention(d_model, num_heads)
        self.residual_conn1 = ResidualConnection(d_model, dropout_rate)
        self.feed_forward = FeedForwardNetwork(d_model, d_ff, dropout_rate)
        self.residual_conn2 = ResidualConnection(d_model, dropout_rate)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        input_to_attention = x
        attention_output = self.masked_self_attention(x, mask)
        x = self.residual_conn1(input_to_attention, attention_output)
        input_to_ffn = x
        ffn_output = self.feed_forward(x)
        output = self.residual_conn2(input_to_ffn, ffn_output)
        return output


class FinalLinearOutput(nn.Module):
    def __init__(self, d_model: int, vocab_size: int):
        super().__init__()
        self.linear = nn.Linear(d_model, vocab_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        logits = self.linear(x)
        return logits


class DecoderOnlyTransformer(nn.Module):
    """
    A complete Decoder-Only Transformer model in PyTorch.

    This model assembles Token Embedding, Positional Encoding, a stack of
    Decoder Blocks, and a final Linear Output layer to perform sequence generation.
    """
    def __init__(
        self,
        vocab_size: int,
        d_model: int,
        num_heads: int,
        num_layers: int,
        d_ff: int,
        max_len: int = 5000,
        dropout_rate: float = 0.1
    ):
        """
        Initializes the DecoderOnlyTransformer model.

        Args:
            vocab_size (int): The total number of unique tokens in the vocabulary.
            d_model (int): The dimensionality of the token embeddings and model's internal representations.
            num_heads (int): The number of attention heads in each DecoderBlock.
            num_layers (int): The number of DecoderBlocks to stack.
            d_ff (int): The hidden layer dimensionality of the FeedForwardNetwork in each DecoderBlock.
            max_len (int): The maximum expected sequence length for positional encoding.
            dropout_rate (float): The dropout probability used across the model.
        """
        super().__init__()

        # 1. Token Embedding Layer: Converts token IDs into dense vectors.
        # Input: (batch_size, sequence_length) -> Output: (batch_size, sequence_length, d_model)
        self.token_embedding = TokenEmbedding(vocab_size=vocab_size, embedding_dim=d_model)

        # 2. Positional Encoding Layer: Adds positional information to token embeddings.
        # Output: (batch_size, sequence_length, d_model)
        self.positional_encoding = PositionalEncoding(d_model=d_model, max_len=max_len)

        self.dropout = nn.Dropout(dropout_rate)

        # 3. Stack of Decoder Blocks: Processes the sequence through multiple layers of
        # masked self-attention and feed-forward networks.
        # Each block maintains the (batch_size, sequence_length, d_model) shape.
        self.decoder_blocks = nn.ModuleList([
            DecoderBlock(d_model, num_heads, d_ff, dropout_rate)
            for _ in range(num_layers)
        ])

        # 4. Final Linear Output Layer: Maps the decoder's output to logit scores over the vocabulary.
        # Input: (batch_size, sequence_length, d_model) -> Output: (batch_size, sequence_length, vocab_size)
        self.final_linear = FinalLinearOutput(d_model=d_model, vocab_size=vocab_size)

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        """
        Performs the forward pass of the DecoderOnlyTransformer model.

        Args:
            token_ids (torch.Tensor): A tensor of input token IDs.
                                      Shape: (batch_size, sequence_length).

        Returns:
            torch.Tensor: A tensor of unnormalized logit scores for each token in the
                          vocabulary at each position.
                          Shape: (batch_size, sequence_length, vocab_size).
        """
        batch_size, sequence_length = token_ids.size()

        # 1. Apply Token Embedding
        # x: (batch_size, sequence_length, d_model)
        x = self.token_embedding(token_ids)

        # 2. Add Positional Encoding
        # x: (batch_size, sequence_length, d_model)
        x = self.positional_encoding(x)

        # Apply dropout to the combined embeddings
        x = self.dropout(x)

        # 3. Create Causal Mask (Look-ahead mask)
        # This mask ensures that each position can only attend to previous positions (including itself).
        # `tril` creates a lower triangular matrix. `bool()` converts it to boolean.
        # `unsqueeze(0).unsqueeze(0)` adds batch and head dimensions for broadcasting with attention scores.
        # mask: (1, 1, sequence_length, sequence_length)
        causal_mask = torch.tril(torch.ones(sequence_length, sequence_length, device=token_ids.device)).bool()
        causal_mask = causal_mask.unsqueeze(0).unsqueeze(0)

        # 4. Pass through Decoder Blocks
        # x: (batch_size, sequence_length, d_model)
        for decoder_block in self.decoder_blocks:
            x = decoder_block(x, mask=causal_mask)

        # 5. Apply Final Linear Output Layer
        # logits: (batch_size, sequence_length, vocab_size)
        logits = self.final_linear(x)

        return logits

# Example Usage and Verification:
# Define model parameters
example_vocab_size = 10000
example_d_model = 256
example_num_heads = 8
example_num_layers = 3 # Stack 3 decoder blocks
example_d_ff = 1024    # Typically 4 * d_model
example_max_len = 500
example_dropout_rate = 0.1

example_batch_size = 2
example_sequence_length = 10

# Instantiate the DecoderOnlyTransformer model
model = DecoderOnlyTransformer(
    vocab_size=example_vocab_size,
    d_model=example_d_model,
    num_heads=example_num_heads,
    num_layers=example_num_layers,
    d_ff=example_d_ff,
    max_len=example_max_len,
    dropout_rate=example_dropout_rate
)
print("DecoderOnlyTransformer model created.")

# Create a sample input tensor of token IDs
# Each token ID should be within [0, vocab_size - 1]
example_token_ids = torch.randint(0, example_vocab_size, (example_batch_size, example_sequence_length))
print(f"\nExample input token IDs shape: {example_token_ids.shape}")

# Perform a forward pass through the model
output_logits = model(example_token_ids)
print(f"Output logits shape: {output_logits.shape}")

# Verify output dimensions match expected
expected_output_shape = (example_batch_size, example_sequence_length, example_vocab_size)
assert output_logits.shape == expected_output_shape
print("Verification successful: Output shape of DecoderOnlyTransformer is correct.")

# You can optionally compute probabilities from logits
# probabilities = torch.softmax(output_logits, dim=-1)
# print(f"Probabilities shape: {probabilities.shape}")
# print(f"Sum of probabilities for the first token in the first sequence: {probabilities[0, 0].sum().item():.4f}")

DecoderOnlyTransformer model created.

Example input token IDs shape: torch.Size([2, 10])
Output logits shape: torch.Size([2, 10, 10000])
Verification successful: Output shape of DecoderOnlyTransformer is correct.


## Final Task

### Subtask:
Provide a summary of the complete decoder-only Transformer implementation and its key components.


## Summary:

### Data Analysis Key Findings

*   A complete decoder-only Transformer model was successfully implemented in PyTorch, integrating token embeddings, sinusoidal positional encodings, multiple decoder blocks, and a final linear output layer.
*   **Token Embedding:** A `TokenEmbedding` layer was created to map `vocab_size=10000` tokens to `d_model=256` dimensional vectors, converting discrete token IDs into dense, continuous representations.
*   **Positional Encoding:** Sinusoidal positional encodings were correctly formulated and implemented to add positional information to `d_model=256` embeddings, allowing the model to process sequences up to a `max_len=500` while retaining order.
*   **Multi-Head Self-Attention:** A `MultiHeadSelfAttention` module was implemented with `d_model=256` and `num_heads=8`, successfully managing query, key, and value projections, scaled dot-product attention, and output projections to jointly attend to different representation subspaces.
*   **Masked Attention:** The `MultiHeadSelfAttention` module was confirmed to support masked attention by applying a mask with $-\infty$ values to future positions in attention scores, which is crucial for enforcing causality in the decoder.
*   **Layer Normalization:** A `LayerNorm` module was implemented to normalize inputs across the feature dimension for each sample. Verification confirmed that the normalized outputs exhibited an approximate mean of 0.0 and variance of 1.0, and included learnable scale ($\gamma$) and shift ($\beta$) parameters.
*   **Feed-Forward Network:** A `FeedForwardNetwork` was implemented with two linear layers, expanding the `d_model=256` input to an intermediate `d_ff=1024` dimension with a ReLU activation, then contracting it back to `d_model=256`, preserving the input's dimensionality.
*   **Residual Connections:** The concept of "Add & Norm" was correctly integrated into a `ResidualConnection` module, applying dropout, adding the sublayer output to its input, and then performing layer normalization, which is essential for training deep networks.
*   **Decoder Block:** Individual `DecoderBlock` units were successfully assembled, combining the masked multi-head self-attention, residual connections, layer normalization, and the feed-forward network in the correct sequence.
*   **Final Linear Output Layer:** A `FinalLinearOutput` layer was implemented to transform the decoder's `d_model=256` output into `vocab_size=10000` unnormalized logit scores, preparing the output for token prediction.
*   **Overall Model Integration:** The `DecoderOnlyTransformer` class successfully orchestrated these components into a coherent architecture. It correctly handled input token IDs, applied positional encoding, processed data through `num_layers=3` decoder blocks (each with `d_model=256`, `num_heads=8`, `d_ff=1024`), generated and applied causal masks, and produced final output logits, all verified by consistent tensor shape transformations.

### Insights or Next Steps

*   The modular design allows for flexible experimentation with different hyperparameters (e.g., number of heads, layers, feed-forward dimensions) or alternative activation functions within the FFN.
*   The next step would be to train this `DecoderOnlyTransformer` on a sequence generation task, such as language modeling, by integrating a suitable loss function (e.g., `torch.nn.CrossEntropyLoss`) and an optimizer.
